In [1]:

import builtins
import os
import sys
from pathlib import Path
import pandas as _winner_pd

_WINNER_SOURCE_DIR = Path('/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/raw_behavior_by_winner')
_WINNER_RESULTS_DIR = Path('/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3')
_WINNER_CSV = Path('/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/all_subjects_final_data.csv')
if str(_WINNER_SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(_WINNER_SOURCE_DIR))
os.chdir(_WINNER_RESULTS_DIR)

def _winner_input(prompt=''):
    prompt_lower = str(prompt).lower()
    if 'csv' in prompt_lower:
        return str(_WINNER_CSV)
    if 'number of bins' in prompt_lower:
        return '4'
    if 'directory to save' in prompt_lower:
        return str(_WINNER_RESULTS_DIR)
    raise RuntimeError(f'Unexpected interactive prompt: {prompt}')

builtins.input = _winner_input
_winner_df = _winner_pd.read_csv(_WINNER_CSV)
print(
    'MODEL-WINNER INPUT:',
    len(_winner_df), 'trials,',
    _winner_df['session_uid'].nunique(), 'sessions, subjects',
    sorted(_winner_df['subject'].unique()),
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as _winner_plt
_winner_plt.show = lambda *args, **kwargs: _winner_plt.close('all')


MODEL-WINNER INPUT: 291 trials, 1 sessions, subjects ['MC06']


# Pooled and grand-average performance

Five subject groupings are shown by row. The left column pools trials; the right column gives equal weight to each session.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from scipy.stats import fisher_exact, wilcoxon

DATA_PATH = Path('all_subjects_final_data.csv')
OUTPUT_PATH = Path('performance_pooled_grand_average.pdf')

SUBJECT_ROWS = [
    ("All subjects", None),
    ('MC06', 'MC06'),
]
CONDITIONS = ['Neutral', 'Neutral\n(no naive)', 'Common', 'Rare']
SHORT_COLOR = '#377eb8'
LONG_COLOR = '#e41a1c'
PAIR_OFFSET = 0.14

CORRECT_OUTCOMES = {'Reward', 'RewardNaive'}
INCORRECT_OUTCOMES = {'Punish', 'PunishNaive'}
VALID_OUTCOMES = CORRECT_OUTCOMES | INCORRECT_OUTCOMES
NON_NAIVE_OUTCOMES = {'Reward', 'Punish'}

def prepare_data(path):
    df = pd.read_csv(path)
    required = {'subject', 'session_idx', 'block_type', 'trial_type', 'outcome'}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    for column in ['session_idx', 'block_type', 'trial_type']:
        df[column] = pd.to_numeric(df[column], errors='coerce')
    df = df[df['outcome'].isin(VALID_OUTCOMES)].copy()
    df['correct'] = df['outcome'].isin(CORRECT_OUTCOMES).astype(float)
    if 'session_uid' in df.columns:
        df['session_key'] = df['session_uid'].astype(str)
    else:
        df['session_key'] = df['subject'].astype(str) + '_S' + df['session_idx'].astype(int).astype(str)
    return df

def condition_mask(df, condition, category):
    short = category == 'Short'
    if condition == 'Neutral':
        return df['block_type'].eq(0) & df['trial_type'].eq(1 if short else 2)
    if condition == 'Neutral (no naive)':
        return (
            df['block_type'].eq(0)
            & df['trial_type'].eq(1 if short else 2)
            & df['outcome'].isin(NON_NAIVE_OUTCOMES)
        )
    if condition == 'Common':
        return (df['block_type'].eq(1) & df['trial_type'].eq(1)) if short else (df['block_type'].eq(2) & df['trial_type'].eq(2))
    if condition == 'Rare':
        return (df['block_type'].eq(2) & df['trial_type'].eq(1)) if short else (df['block_type'].eq(1) & df['trial_type'].eq(2))
    raise ValueError(f'Unknown condition: {condition}')

def subset(df, condition, category):
    return df.loc[condition_mask(df, condition, category)].copy()

def pooled_stats(data):
    n = len(data)
    if n == 0:
        return np.nan, np.nan, 0
    mean = data['correct'].mean()
    sem = np.sqrt(mean * (1 - mean) / n)
    return mean, sem, n

def grand_stats(data):
    session_means = data.groupby('session_key')['correct'].mean()
    if session_means.empty:
        return np.nan, np.nan, 0, session_means
    sem = session_means.sem() if len(session_means) > 1 else 0.0
    return session_means.mean(), sem, len(data), session_means

def pooled_pvalue(short_data, long_data):
    if short_data.empty or long_data.empty:
        return np.nan
    table = [
        [int(short_data['correct'].sum()), int(len(short_data) - short_data['correct'].sum())],
        [int(long_data['correct'].sum()), int(len(long_data) - long_data['correct'].sum())],
    ]
    return fisher_exact(table, alternative='two-sided').pvalue

def grand_pvalue(short_data, long_data):
    paired = pd.concat(
        [short_data.groupby('session_key')['correct'].mean().rename('short'),
         long_data.groupby('session_key')['correct'].mean().rename('long')],
        axis=1, join='inner'
    ).dropna()
    if len(paired) < 2:
        return np.nan
    differences = paired['short'] - paired['long']
    if np.allclose(differences, 0):
        return 1.0
    try:
        return wilcoxon(paired['short'], paired['long'], alternative='two-sided').pvalue
    except ValueError:
        return np.nan

def significance_label(p):
    if not np.isfinite(p):
        return 'n/a'
    if p < 0.0001:
        return '****'
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return 'ns'

def style_axis(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(False)
    ax.set_ylabel('Performance (mean ± SEM)')
    ax.set_xticks(np.arange(len(CONDITIONS)))
    ax.set_xticklabels(CONDITIONS)

def add_significance(ax, x, left_top, right_top, label):
    if not np.isfinite(left_top) or not np.isfinite(right_top):
        return
    y = max(left_top, right_top) + 0.075
    x1, x2 = x - PAIR_OFFSET, x + PAIR_OFFSET
    ax.plot([x1, x1, x2, x2], [y - 0.015, y, y, y - 0.015], color='black', linewidth=1)
    ax.text(x, y + 0.012, label, ha='center', va='bottom', fontsize=9)

def plot_panel(ax, data, mode, title):
    trial_counts = {'Short': 0, 'Long': 0}
    condition_keys = ['Neutral', 'Neutral (no naive)', 'Common', 'Rare']
    for x, condition in enumerate(condition_keys):
        short_data = subset(data, condition, 'Short')
        long_data = subset(data, condition, 'Long')
        trial_counts['Short'] += len(short_data)
        trial_counts['Long'] += len(long_data)
        if mode == 'pooled':
            sm, se, _ = pooled_stats(short_data)
            lm, le, _ = pooled_stats(long_data)
            p = pooled_pvalue(short_data, long_data)
        else:
            sm, se, _, _ = grand_stats(short_data)
            lm, le, _, _ = grand_stats(long_data)
            p = grand_pvalue(short_data, long_data)
        ax.errorbar(x - PAIR_OFFSET, sm, yerr=se, fmt='o', color=SHORT_COLOR,
                    markersize=6, capsize=4, elinewidth=1.5)
        ax.errorbar(x + PAIR_OFFSET, lm, yerr=le, fmt='o', color=LONG_COLOR,
                    markersize=6, capsize=4, elinewidth=1.5)
        add_significance(ax, x, sm + se, lm + le, significance_label(p))
    style_axis(ax)
    ax.set_title(title, fontsize=11)
    handles = [
        Line2D([0], [0], marker='o', linestyle='none', color=SHORT_COLOR,
               label=f"Short (n={trial_counts['Short']:,} trials)"),
        Line2D([0], [0], marker='o', linestyle='none', color=LONG_COLOR,
               label=f"Long (n={trial_counts['Long']:,} trials)"),
    ]
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.01, 1.0),
              frameon=True, borderaxespad=0, fontsize=8)

def make_figure(df, output_path=OUTPUT_PATH):
    fig = plt.figure(figsize=(22, 27))
    gs = gridspec.GridSpec(5, 2, figure=fig, hspace=0.55, wspace=0.58)
    for row, (label, subject) in enumerate(SUBJECT_ROWS):
        row_data = df if subject is None else df[df['subject'].eq(subject)].copy()
        if row_data.empty:
            raise ValueError(f'No rows found for subject {subject}')
        plot_panel(fig.add_subplot(gs[row, 0]), row_data, 'pooled', f'{label} — pooled trials')
        plot_panel(fig.add_subplot(gs[row, 1]), row_data, 'grand', f'{label} — grand average across sessions')
    fig.suptitle('Performance by trial category', fontsize=16, y=0.995)
    fig.subplots_adjust(left=0.07, right=0.88, top=0.975, bottom=0.04)
    output_path = Path(output_path)
    fig.savefig(output_path, format='pdf', bbox_inches='tight')
    plt.show()
    return output_path

df = prepare_data(DATA_PATH)
saved_path = make_figure(df)
print(f'Saved: {saved_path.resolve()}')


Saved: /home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/performance_pooled_grand_average.pdf


In [3]:
# Within-block rare-trial and adjusted-transition performance changes
from scipy.stats import mannwhitneyu, wilcoxon

BLOCK_CHANGE_OUTPUT_PATH = Path('block_segment_performance_changes.pdf')
CHANGE_COLORS = {
    'Transition': '#6a3d9a',
    'Part 1': '#1b9e77',
    'Part 2': '#d95f02',
    'Part 3': '#7570b3',
}

def block_change_prepare_data(path=DATA_PATH):
    data = pd.read_csv(path).copy()
    required = {'subject', 'block_type', 'trial_type', 'outcome'}
    missing = required.difference(data.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    data['_source_order'] = np.arange(len(data))
    for column in ['block_type', 'trial_type', 'warm_up']:
        if column in data.columns:
            data[column] = pd.to_numeric(data[column], errors='coerce')
    if 'warm_up' in data.columns:
        data = data[data['warm_up'].fillna(0).eq(0)].copy()
    if 'session_uid' in data.columns:
        data['change_session'] = data['session_uid'].astype(str)
    elif {'subject', 'subject_session_idx'}.issubset(data.columns):
        data['change_session'] = data['subject'].astype(str) + '_S' + data['subject_session_idx'].astype(str)
    else:
        data['change_session'] = data['subject'].astype(str) + '_S' + data['session_idx'].astype(str)
    data['correct'] = np.nan
    data.loc[data['outcome'].isin(CORRECT_OUTCOMES), 'correct'] = 1.0
    data.loc[data['outcome'].isin(INCORRECT_OUTCOMES), 'correct'] = 0.0
    data['is_rare'] = (
        (data['block_type'].eq(1) & data['trial_type'].eq(2))
        | (data['block_type'].eq(2) & data['trial_type'].eq(1))
    )
    return data.sort_values(['change_session', '_source_order']).copy()

def adjusted_block_start(sdf, formal_switch_idx):
    # Match the block-aligned analysis: move a formal switch backward to the
    # first trial in a terminal run of rare trials from the preceding block.
    last_previous = formal_switch_idx - 1
    if last_previous < 0 or not bool(sdf.loc[last_previous, 'is_rare']):
        return formal_switch_idx
    previous_block = sdf.loc[last_previous, 'block_type']
    start = last_previous
    while start - 1 >= 0:
        previous_row = sdf.loc[start - 1]
        if previous_row['block_type'] != previous_block or not bool(previous_row['is_rare']):
            break
        start -= 1
    return start

def qualified_change(sdf, trial_zero):
    if trial_zero < 2:
        return np.nan
    trial_minus_1 = sdf.loc[trial_zero - 1]
    trial_minus_2 = sdf.loc[trial_zero - 2]
    trial_0 = sdf.loc[trial_zero]
    # Trial -1 must be neither rare nor immediately post-rare.
    if bool(trial_minus_1['is_rare']) or bool(trial_minus_2['is_rare']):
        return np.nan
    if pd.isna(trial_minus_1['correct']) or pd.isna(trial_0['correct']):
        return np.nan
    return float(trial_0['correct'] - trial_minus_1['correct'])

def collect_block_change_events(data):
    events = []
    for session_key, session_df in data.groupby('change_session', sort=False):
        # Neutral trials are outside the short/long block sequence.
        sdf = session_df[session_df['block_type'].isin([1, 2])].copy()
        sdf = sdf.sort_values('_source_order').reset_index(drop=True)
        if sdf.empty:
            continue
        formal_switches = sdf.index[
            sdf['block_type'].ne(sdf['block_type'].shift())
            & sdf['block_type'].shift().notna()
        ].tolist()
        adjusted_starts = [0] + [adjusted_block_start(sdf, idx) for idx in formal_switches]
        adjusted_starts = list(dict.fromkeys(adjusted_starts))
        invalid_starts = [
            start for start in adjusted_starts[1:]
            if start > 0 and bool(sdf.loc[start - 1, 'is_rare'])
        ]
        if invalid_starts:
            raise AssertionError(
                f'Adjusted transition preceded by a rare trial in {session_key}: {invalid_starts}'
            )
        if len(adjusted_starts) < 2:
            continue

        # Every interval is [start, next start); the final session block is omitted.
        retained_blocks = list(zip(adjusted_starts[:-1], adjusted_starts[1:]))
        for block_number, (start, stop) in enumerate(retained_blocks):
            positions = np.arange(start, stop, dtype=int)
            for part_number, part_positions in enumerate(np.array_split(positions, 3), start=1):
                for anchor_idx in part_positions:
                    if not bool(sdf.loc[anchor_idx, 'is_rare']):
                        continue
                    change = qualified_change(sdf, int(anchor_idx))
                    if np.isfinite(change):
                        events.append({
                            'subject': sdf.loc[anchor_idx, 'subject'],
                            'session': session_key,
                            'block_id': f'{session_key}_B{block_number}',
                            'kind': f'Part {part_number}',
                            'block_context': ('Short block' if sdf.loc[anchor_idx, 'block_type'] == 1 else 'Long block'),
                            'transition_direction': None,
                            'change': change,
                        })

        # Transition into the excluded final block is also excluded.
        for transition_number, transition_idx in enumerate(adjusted_starts[1:-1], start=1):
            change = qualified_change(sdf, int(transition_idx))
            if np.isfinite(change):
                formal_switch_idx = next(
                    idx for idx in formal_switches
                    if adjusted_block_start(sdf, idx) == transition_idx
                )
                from_block = int(sdf.loc[formal_switch_idx - 1, 'block_type'])
                to_block = int(sdf.loc[formal_switch_idx, 'block_type'])
                transition_direction = (
                    'Short to Long' if (from_block, to_block) == (1, 2)
                    else 'Long to Short' if (from_block, to_block) == (2, 1)
                    else None
                )
                events.append({
                    'subject': sdf.loc[transition_idx, 'subject'],
                    'session': session_key,
                    'block_id': f'{session_key}_T{transition_number}',
                    'kind': 'Transition',
                    'block_context': None,
                    'transition_direction': transition_direction,
                    'change': change,
                })
    return pd.DataFrame(events)

def change_values(events, kind, mode):
    selected = events[events['kind'].eq(kind)].copy()
    if mode == 'pooled':
        return selected['change'].dropna(), int(len(selected))
    values = selected.groupby('session')['change'].mean().dropna()
    return values, int(len(selected))

def change_mean_sem(values):
    values = pd.Series(values).dropna()
    if values.empty:
        return np.nan, np.nan
    return float(values.mean()), float(values.sem()) if len(values) > 1 else 0.0

def change_pvalue(events, left_kind, right_kind, mode):
    if mode == 'pooled':
        left = events.loc[events['kind'].eq(left_kind), 'change'].dropna()
        right = events.loc[events['kind'].eq(right_kind), 'change'].dropna()
        if left.empty or right.empty:
            return np.nan
        return mannwhitneyu(left, right, alternative='two-sided').pvalue
    session_means = events.groupby(['session', 'kind'])['change'].mean().unstack()
    if left_kind not in session_means or right_kind not in session_means:
        return np.nan
    paired = session_means[[left_kind, right_kind]].dropna()
    if len(paired) < 2:
        return np.nan
    difference = paired[left_kind] - paired[right_kind]
    if np.allclose(difference, 0):
        return 1.0
    try:
        return wilcoxon(paired[left_kind], paired[right_kind]).pvalue
    except ValueError:
        return np.nan

def change_sig_label(p):
    if not np.isfinite(p): return 'n/a'
    if p < 0.0001: return '****'
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    return 'ns'

def add_change_bracket(ax, x1, x2, y, label):
    ax.plot([x1, x1, x2, x2], [y - 0.025, y, y, y - 0.025], color='black', linewidth=0.9)
    ax.text((x1 + x2) / 2, y + 0.015, label, ha='center', va='bottom', fontsize=8)

def plot_change_panel(ax, events, mode, title):
    positions = {
        ('Transition', 0): -0.14, ('Part 1', 0): 0.14,
        ('Part 2', 1): 0.86, ('Transition', 1): 1.14,
        ('Part 3', 2): 1.86, ('Transition', 2): 2.14,
        ('Part 1', 3): 2.72, ('Part 2', 3): 3.00, ('Part 3', 3): 3.28,
    }
    stats = {}
    counts = {}
    for kind in ['Transition', 'Part 1', 'Part 2', 'Part 3']:
        values, counts[kind] = change_values(events, kind, mode)
        stats[kind] = change_mean_sem(values)
    for (kind, tick), x in positions.items():
        mean, sem = stats[kind]
        ax.errorbar(x, mean, yerr=sem, fmt='o', color=CHANGE_COLORS[kind],
                    markersize=5.5, capsize=3.5, elinewidth=1.3)
    pair_specs = [
        ('Transition', 'Part 1', -0.14, 0.14, 1.08),
        ('Part 2', 'Transition', 0.86, 1.14, 1.08),
        ('Part 3', 'Transition', 1.86, 2.14, 1.08),
        ('Part 1', 'Part 2', 2.72, 3.00, 1.04),
        ('Part 2', 'Part 3', 3.00, 3.28, 1.25),
        ('Part 1', 'Part 3', 2.72, 3.28, 1.46),
    ]
    for left, right, x1, x2, y in pair_specs:
        add_change_bracket(ax, x1, x2, y, change_sig_label(change_pvalue(events, left, right, mode)))
    ax.axhline(0, color='0.45', linestyle=':', linewidth=1)
    ax.set_xlim(-0.5, 3.55)
    ax.set_xticks(range(4))
    ax.set_xticklabels(['Transition vs\nPart 1', 'Part 2 vs\nTransition',
                        'Part 3 vs\nTransition', 'Parts 1, 2, 3'])
    ax.set_ylabel('Performance change\n(trial 0 − trial −1)')
    ax.set_title(title, fontsize=10)
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    handles = [Line2D([0], [0], marker='o', linestyle='none', color=CHANGE_COLORS[k],
                      label=f'{k} (n={counts[k]:,} events)')
               for k in ['Transition', 'Part 1', 'Part 2', 'Part 3']]
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.01, 1),
              frameon=True, fontsize=7, borderaxespad=0)

def contextual_change_events(events, block_context, transition_direction):
    return events[
        (events['kind'].eq('Transition') & events['transition_direction'].eq(transition_direction))
        | (events['kind'].ne('Transition') & events['block_context'].eq(block_context))
    ].copy()

def build_block_change_page(events):
    fig = plt.figure(figsize=(66, 27))
    gs = gridspec.GridSpec(5, 6, figure=fig, hspace=0.56, wspace=0.62)
    for row, (row_label, subject) in enumerate(SUBJECT_ROWS):
        row_events = events if subject is None else events[events['subject'].eq(subject)].copy()
        plot_change_panel(fig.add_subplot(gs[row, 0]), row_events, 'pooled', f'{row_label} — pooled events')
        plot_change_panel(fig.add_subplot(gs[row, 1]), row_events, 'grand', f'{row_label} — grand average across sessions')
        short_events = contextual_change_events(row_events, 'Short block', 'Short to Long')
        plot_change_panel(fig.add_subplot(gs[row, 2]), short_events, 'pooled', f'{row_label} — short-block rare + short-to-long transition — pooled events')
        plot_change_panel(fig.add_subplot(gs[row, 3]), short_events, 'grand', f'{row_label} — short-block rare + short-to-long transition — grand average')
        long_events = contextual_change_events(row_events, 'Long block', 'Long to Short')
        plot_change_panel(fig.add_subplot(gs[row, 4]), long_events, 'pooled', f'{row_label} — long-block rare + long-to-short transition — pooled events')
        plot_change_panel(fig.add_subplot(gs[row, 5]), long_events, 'grand', f'{row_label} — long-block rare + long-to-short transition — grand average')
    fig.suptitle('Qualified rare-trial and adjusted block-transition performance changes', fontsize=16, y=0.995)
    fig.subplots_adjust(left=0.025, right=0.96, top=0.975, bottom=0.045)
    return fig

from block_sequence_epoch_analysis import make_extended_figure

block_change_data = block_change_prepare_data()
block_change_events = collect_block_change_events(block_change_data)
(block_change_pdf_path, block_sequence_events, block_sequence_window_events,
 block_pre_rare_events, block_pre_rare_window_events,
 block_double_rare_events, block_double_rare_window_events,
 block_consecutive_common_events, block_consecutive_common_window_events,
 block_previous_outcome_events, block_previous_outcome_window_events) = make_extended_figure(
    original_page_builder=build_block_change_page,
    change_events=block_change_events,
    data=block_change_data,
    adjusted_block_start=adjusted_block_start,
    subject_rows=SUBJECT_ROWS,
    output_path=BLOCK_CHANGE_OUTPUT_PATH,
)
print(f'Saved: {block_change_pdf_path.resolve()}')
print(block_change_events.groupby(['subject', 'kind']).size().unstack(fill_value=0))
print('\nQualified common/common/rare/common sequences:')
print(block_sequence_events.groupby(['subject', 'epoch']).size().unstack(fill_value=0))
print(f'Qualified common/common/rare sequences: {len(block_pre_rare_events):,}')
print(f'Qualified common/common/rare/rare/common sequences: {len(block_double_rare_events):,}')
print(f'Consecutive-common targets: {len(block_consecutive_common_events):,}')
print(f'Previous-outcome common targets: {len(block_previous_outcome_events):,}')


Saved: /home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/block_segment_performance_changes.pdf
kind     Part 1  Part 2  Part 3  Transition
subject                                    
MC06         11       8      10           5

Qualified common/common/rare/common sequences:
epoch    Epoch 1  Epoch 2  Epoch 3
subject                           
MC06           6        7        9
Qualified common/common/rare sequences: 27
Qualified common/common/rare/rare/common sequences: 4
Consecutive-common targets: 62
Previous-outcome common targets: 140


In [4]:
# Early/late easy-versus-hard rare-trial performance changes
from rare_difficulty_epoch_analysis import run as run_rare_difficulty_epoch_analysis

rare_difficulty_pdf_path, rare_difficulty_events = run_rare_difficulty_epoch_analysis(
    data=block_change_data,
    adjusted_block_start=adjusted_block_start,
    qualified_change=qualified_change,
    subject_rows=SUBJECT_ROWS,
    show=True,
)


Saved: /home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/rare_difficulty_early_late_performance_changes.pdf
subject  block_context  epoch  difficulty
MC06     Long block     Early  Easy          1
                               Hard          3
                        Late   Easy          1
                               Hard          1
         Short block    Early  Easy          3
                        Late   Hard          1
dtype: int64


In [5]:
# Choice bias relative to the matching non-naive neutral-block baseline
BIAS_OUTPUT_PATH = Path('choice_bias_relative_to_neutral.pdf')
BIAS_CONDITIONS = [
    ('Common', 'Easy'),
    ('Common', 'Difficult'),
    ('Rare', 'Easy'),
    ('Rare', 'Difficult'),
]
BIAS_LABELS = ['Common\neasy', 'Common\ndifficult', 'Rare\neasy', 'Rare\ndifficult']
BIAS_COLORS = ['#1b9e77', '#66a61e', '#d95f02', '#e6ab02']
NEUTRAL_COLOR = '#7570b3'
BLOCK_COLOR = '#e7298a'
from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import shapiro, ttest_1samp

def bias_prepare_data(path=DATA_PATH):
    data = pd.read_csv(path).copy()
    required = {'subject', 'block_type', 'trial_type', 'trial_isi', 'outcome', 'lick_side'}
    missing = required.difference(data.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    for column in ['block_type', 'trial_type', 'trial_isi']:
        data[column] = pd.to_numeric(data[column], errors='coerce')
    # Naive outcomes are excluded from both the neutral reference and block trials.
    data = data[data['outcome'].isin(NON_NAIVE_OUTCOMES)].copy()
    # Match the psychometric figures: Right is long-associated for the normal
    # contingency, while Left is long-associated for the reverse contingency.
    choice = data['lick_side'].astype(str).str.strip().str.lower()
    if 'Contingency' in data.columns:
        contingency = data['Contingency'].astype(str)
        normal = contingency.str.contains('Normal - Short/Left', na=False)
        reverse = contingency.str.contains('Reverse - Long/Left', na=False)
    elif 'Contingency_idx' in data.columns:
        normal = pd.to_numeric(data['Contingency_idx'], errors='coerce').eq(1)
        reverse = pd.to_numeric(data['Contingency_idx'], errors='coerce').eq(2)
    else:
        normal = pd.Series(True, index=data.index)
        reverse = pd.Series(False, index=data.index)
    unknown = ~(normal | reverse)
    normal = normal | unknown  # Same fallback used by the psychometric notebook.
    data['choose_long'] = np.nan
    data.loc[normal, 'choose_long'] = choice[normal].eq('right').astype(float)
    data.loc[reverse, 'choose_long'] = choice[reverse].eq('left').astype(float)
    data = data.dropna(subset=['block_type', 'trial_type', 'trial_isi', 'choose_long']).copy()
    if 'session_uid' in data.columns:
        data['bias_session'] = data['session_uid'].astype(str)
    elif {'subject', 'subject_session_idx'}.issubset(data.columns):
        data['bias_session'] = (
            data['subject'].astype(str) + '_S' + data['subject_session_idx'].astype(str)
        )
    else:
        data['bias_session'] = data['subject'].astype(str) + '_S' + data['session_idx'].astype(str)
    return data

def bias_session_extremes(session_data):
    short = session_data.loc[session_data['trial_type'].eq(1), 'trial_isi'].dropna()
    long = session_data.loc[session_data['trial_type'].eq(2), 'trial_isi'].dropna()
    return {
        (1, 'Easy'): short.min() if len(short) else np.nan,
        (1, 'Difficult'): short.max() if len(short) else np.nan,
        (2, 'Easy'): long.max() if len(long) else np.nan,
        (2, 'Difficult'): long.min() if len(long) else np.nan,
    }

def bias_block_mask(data, condition, trial_type):
    if condition == 'Common':
        block_type = 1 if trial_type == 1 else 2
    elif condition == 'Rare':
        block_type = 2 if trial_type == 1 else 1
    else:
        raise ValueError(f'Unknown condition: {condition}')
    return data['block_type'].eq(block_type) & data['trial_type'].eq(trial_type)

def bias_add_psychometric_bins(data, n_bins_per_type=4):
    """Assign the same seeded, trial-type-specific qcut bins used by psychometrics."""
    binned = data.copy()
    binned['psychometric_bin'] = np.nan
    binned['binned_difficulty'] = pd.NA
    for _, session_data in binned.groupby('bias_session', sort=False):
        for trial_type in [1, 2]:
            trial_index = session_data.index[session_data['trial_type'].eq(trial_type)]
            if len(trial_index) == 0:
                continue
            # This intentionally matches get_binned_stats in psychometric_analysis.ipynb.
            np.random.seed(42)
            jittered_isi = (
                binned.loc[trial_index, 'trial_isi']
                + np.random.normal(0, 1e-6, len(trial_index))
            )
            try:
                bin_codes = pd.qcut(
                    jittered_isi, q=n_bins_per_type, labels=False, duplicates='drop'
                )
            except ValueError:
                continue
            if bin_codes.isna().all():
                continue
            binned.loc[trial_index, 'psychometric_bin'] = bin_codes.to_numpy(dtype=float)
            lowest_bin = float(bin_codes.min())
            highest_bin = float(bin_codes.max())
            if np.isclose(lowest_bin, highest_bin):
                continue
            # Short: farthest below the boundary is easy; nearest is hard.
            # Long: farthest above the boundary is easy; nearest is hard.
            easy_bin, difficult_bin = (
                (lowest_bin, highest_bin) if trial_type == 1
                else (highest_bin, lowest_bin)
            )
            binned.loc[trial_index[bin_codes.to_numpy() == easy_bin], 'binned_difficulty'] = 'Easy'
            binned.loc[trial_index[bin_codes.to_numpy() == difficult_bin], 'binned_difficulty'] = 'Difficult'
    return binned

def collect_binned_bias_components(data, n_bins_per_type=4):
    binned = bias_add_psychometric_bins(data, n_bins_per_type=n_bins_per_type)
    rows = []
    for session, session_data in binned.groupby('bias_session', sort=False):
        subject = session_data['subject'].iloc[0]
        for condition, level in BIAS_CONDITIONS:
            for trial_type in [1, 2]:
                level_mask = session_data['binned_difficulty'].eq(level).fillna(False)
                block_trials = session_data.loc[
                    bias_block_mask(session_data, condition, trial_type) & level_mask
                ]
                neutral_trials = session_data.loc[
                    session_data['block_type'].eq(0)
                    & session_data['trial_type'].eq(trial_type)
                    & level_mask
                ]
                if block_trials.empty or neutral_trials.empty:
                    continue
                block_p = float(block_trials['choose_long'].mean())
                neutral_p = float(neutral_trials['choose_long'].mean())
                rows.append({
                    'subject': subject, 'session': session, 'condition': condition,
                    'difficulty': level, 'trial_type': trial_type,
                    'block_context': (
                        'Short block'
                        if (condition == 'Common' and trial_type == 1)
                        or (condition == 'Rare' and trial_type == 2)
                        else 'Long block'
                    ),
                    'block_p_long': block_p, 'neutral_p_long': neutral_p,
                    'bias': block_p - neutral_p,
                    'block_success': int(block_trials['choose_long'].sum()),
                    'neutral_success': int(neutral_trials['choose_long'].sum()),
                    'block_n': len(block_trials), 'neutral_n': len(neutral_trials),
                })
    return pd.DataFrame(rows)

def collect_bias_bin_curve_components(data, n_bins_per_type=4):
    """Collect block-minus-neutral contrasts for every psychometric ISI bin."""
    binned = bias_add_psychometric_bins(data, n_bins_per_type=n_bins_per_type)
    rows = []
    for session, session_data in binned.groupby('bias_session', sort=False):
        subject = session_data['subject'].iloc[0]
        for condition in ['Common', 'Rare']:
            for trial_type in [1, 2]:
                bin_values = sorted(
                    session_data.loc[
                        session_data['trial_type'].eq(trial_type), 'psychometric_bin'
                    ].dropna().unique()
                )
                for bin_index in bin_values:
                    bin_mask = session_data['psychometric_bin'].eq(bin_index)
                    block_trials = session_data.loc[
                        bias_block_mask(session_data, condition, trial_type) & bin_mask
                    ]
                    neutral_trials = session_data.loc[
                        session_data['block_type'].eq(0)
                        & session_data['trial_type'].eq(trial_type)
                        & bin_mask
                    ]
                    if block_trials.empty or neutral_trials.empty:
                        continue
                    block_p = float(block_trials['choose_long'].mean())
                    neutral_p = float(neutral_trials['choose_long'].mean())
                    matched_isi = pd.concat([
                        block_trials['trial_isi'], neutral_trials['trial_isi']
                    ], ignore_index=True)
                    rows.append({
                        'subject': subject, 'session': session, 'condition': condition,
                        'trial_type': trial_type, 'bin_index': int(bin_index),
                        'isi_mean': float(matched_isi.mean()),
                        'block_context': (
                            'Short block'
                            if (condition == 'Common' and trial_type == 1)
                            or (condition == 'Rare' and trial_type == 2)
                            else 'Long block'
                        ),
                        'block_p_long': block_p, 'neutral_p_long': neutral_p,
                        'bias': block_p - neutral_p,
                        'block_success': int(block_trials['choose_long'].sum()),
                        'neutral_success': int(neutral_trials['choose_long'].sum()),
                        'block_n': len(block_trials), 'neutral_n': len(neutral_trials),
                    })
    return pd.DataFrame(rows)

def collect_bias_components(data):
    rows = []
    for session, session_data in data.groupby('bias_session', sort=False):
        limits = bias_session_extremes(session_data)
        subject = session_data['subject'].iloc[0]
        for condition, level in BIAS_CONDITIONS:
            for trial_type in [1, 2]:
                target_isi = limits[(trial_type, level)]
                opposite_isi = limits[(trial_type, 'Difficult' if level == 'Easy' else 'Easy')]
                # A one-level session cannot distinguish easy from difficult.
                if not np.isfinite(target_isi) or np.isclose(target_isi, opposite_isi, rtol=0, atol=1e-9):
                    continue
                isi_mask = np.isclose(session_data['trial_isi'], target_isi, rtol=0, atol=1e-9)
                block_trials = session_data.loc[
                    bias_block_mask(session_data, condition, trial_type) & isi_mask
                ]
                neutral_trials = session_data.loc[
                    session_data['block_type'].eq(0)
                    & session_data['trial_type'].eq(trial_type)
                    & isi_mask
                ]
                if block_trials.empty or neutral_trials.empty:
                    continue
                block_p = float(block_trials['choose_long'].mean())
                neutral_p = float(neutral_trials['choose_long'].mean())
                rows.append({
                    'subject': subject, 'session': session, 'condition': condition,
                    'difficulty': level, 'trial_type': trial_type,
                    'block_context': (
                        'Short block'
                        if (condition == 'Common' and trial_type == 1)
                        or (condition == 'Rare' and trial_type == 2)
                        else 'Long block'
                    ),
                    'block_p_long': block_p, 'neutral_p_long': neutral_p,
                    'bias': block_p - neutral_p,
                    'block_success': int(block_trials['choose_long'].sum()),
                    'neutral_success': int(neutral_trials['choose_long'].sum()),
                    'block_n': len(block_trials), 'neutral_n': len(neutral_trials),
                })
    return pd.DataFrame(rows)

def bias_pooled_stats(components):
    if components.empty:
        return np.nan, np.nan, 0, 0
    # Pool session-by-stimulus contrasts, weighting by the number of block trials.
    weights = components['block_n'].to_numpy(dtype=float)
    weights = weights / weights.sum()
    mean = float(np.sum(weights * components['bias'].to_numpy(dtype=float)))
    block_var = (
        components['block_p_long'] * (1 - components['block_p_long']) / components['block_n']
    )
    neutral_var = (
        components['neutral_p_long'] * (1 - components['neutral_p_long']) / components['neutral_n']
    )
    sem = float(np.sqrt(np.sum(weights ** 2 * (block_var + neutral_var))))
    return mean, sem, int(components['block_n'].sum()), components['session'].nunique()

def bias_grand_stats(components):
    if components.empty:
        return np.nan, np.nan, 0, pd.Series(dtype=float)
    # Short- and long-stimulus contrasts receive equal weight within a session.
    session_bias = components.groupby('session')['bias'].mean()
    sem = float(session_bias.sem()) if len(session_bias) > 1 else 0.0
    return float(session_bias.mean()), sem, len(session_bias), session_bias

def bias_condition_components(components, condition, level):
    return components[
        components['condition'].eq(condition) & components['difficulty'].eq(level)
    ].copy()

def bias_distribution_values(components, mode):
    if components.empty:
        return pd.Series(dtype=float)
    if mode == 'pooled':
        # Independent pooled units are session-by-stimulus bias contrasts.
        return components['bias'].dropna().astype(float)
    # Grand-average units give every session one equally weighted bias value.
    return components.groupby('session')['bias'].mean().dropna().astype(float)

def bias_zero_test(components, mode, normality_alpha=0.01):
    values = bias_distribution_values(components, mode)
    if values.empty:
        return {'values': values, 'center': np.nan, 'lower': np.nan, 'upper': np.nan,
                'normality_p': np.nan, 'non_normal': False, 'test': 'n/a', 'p': np.nan}
    normality_p = shapiro(values).pvalue if len(values) >= 3 else np.nan
    non_normal = bool(np.isfinite(normality_p) and normality_p < normality_alpha)
    if non_normal:
        center = float(values.median())
        q1, q3 = values.quantile([0.25, 0.75]).to_numpy(float)
        if np.allclose(values, 0):
            p = 1.0
        else:
            try:
                p = wilcoxon(values, alternative='two-sided').pvalue
            except ValueError:
                p = np.nan
        return {'values': values, 'center': center, 'lower': center - q1, 'upper': q3 - center,
                'normality_p': normality_p, 'non_normal': True, 'test': 'Wilcoxon', 'p': p}
    center = float(values.mean())
    sem = float(values.sem()) if len(values) > 1 else 0.0
    p = ttest_1samp(values, 0.0, nan_policy='omit').pvalue if len(values) > 1 else np.nan
    return {'values': values, 'center': center, 'lower': sem, 'upper': sem,
            'normality_p': normality_p, 'non_normal': False, 'test': 'one-sample t', 'p': p}

def plot_bias_panel(ax, components, mode, title):
    for x, ((condition, level), color) in enumerate(zip(BIAS_CONDITIONS, BIAS_COLORS)):
        selected = bias_condition_components(components, condition, level)
        session_n = selected['session'].nunique() if not selected.empty else 0
        trial_n = int(selected['block_n'].sum()) if not selected.empty else 0
        if mode == 'pooled':
            # True trial-pooled contrast, consistent with the pooled pair page.
            neutral_mean, neutral_sem, block_mean, block_sem, p_value, _ = bias_pair_stats(
                selected, mode
            )
            mean = block_mean - neutral_mean
            sem = float(np.sqrt(block_sem ** 2 + neutral_sem ** 2))
        else:
            # Equal-session-weighted grand average and paired test versus zero.
            stats = bias_zero_test(selected, mode)
            values = stats['values']
            mean = float(values.mean()) if len(values) else np.nan
            sem = float(values.sem()) if len(values) > 1 else 0.0
            p_value = stats['p']
        ax.errorbar(x, mean, yerr=sem, fmt='o', color=color, markersize=7,
                    capsize=4, elinewidth=1.5)
        if np.isfinite(mean):
            star_y = mean + sem + 0.018
            ax.text(x, star_y, bias_significance_label(p_value), ha='center', va='bottom',
                    fontsize=8, fontweight='bold')
        ax.text(x, -0.18, f'{session_n} sess\n{trial_n:,} trials', ha='center', va='top',
                fontsize=7, transform=ax.get_xaxis_transform(), clip_on=False)
    ax.axhline(0, color='0.45', linestyle=':', linewidth=1)
    ax.set_xlim(-0.5, 3.5)
    ax.margins(y=0.22)
    ax.set_xticks(range(4), BIAS_LABELS)
    ax.set_ylabel('Choice bias: ΔP(Choose Long-Associated Response)\n(block − matched neutral)')
    ax.set_title(title, fontsize=11)
    ax.text(0.01, 0.02, '○ mean ± SEM', transform=ax.transAxes, fontsize=6)
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

def bias_significance_label(p):
    if not np.isfinite(p): return 'n/a'
    if p < 0.0001: return '****'
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    return 'ns'

def bias_pair_stats(components, mode):
    if components.empty:
        return np.nan, np.nan, np.nan, np.nan, np.nan, 0
    if mode == 'pooled':
        block_n = int(components['block_n'].sum())
        neutral_n = int(components['neutral_n'].sum())
        block_success = int(components['block_success'].sum())
        neutral_success = int(components['neutral_success'].sum())
        block_mean = block_success / block_n
        neutral_mean = neutral_success / neutral_n
        block_sem = np.sqrt(block_mean * (1 - block_mean) / block_n)
        neutral_sem = np.sqrt(neutral_mean * (1 - neutral_mean) / neutral_n)
        table = [[block_success, block_n - block_success],
                 [neutral_success, neutral_n - neutral_success]]
        p = fisher_exact(table, alternative='two-sided').pvalue
        return neutral_mean, neutral_sem, block_mean, block_sem, p, components['session'].nunique()
    paired = components.groupby('session').agg(
        neutral=('neutral_p_long', 'mean'), block=('block_p_long', 'mean')
    ).dropna()
    if paired.empty:
        return np.nan, np.nan, np.nan, np.nan, np.nan, 0
    neutral_mean = float(paired['neutral'].mean())
    block_mean = float(paired['block'].mean())
    neutral_sem = float(paired['neutral'].sem()) if len(paired) > 1 else 0.0
    block_sem = float(paired['block'].sem()) if len(paired) > 1 else 0.0
    difference = paired['block'] - paired['neutral']
    if len(paired) < 2:
        p = np.nan
    elif np.allclose(difference, 0):
        p = 1.0
    else:
        try:
            p = wilcoxon(paired['block'], paired['neutral'], alternative='two-sided').pvalue
        except ValueError:
            p = np.nan
    return neutral_mean, neutral_sem, block_mean, block_sem, p, len(paired)

def add_bias_pair_bracket(ax, x1, x2, top, p):
    gap, height = 0.035, 0.018
    y = top + gap
    ax.plot([x1, x1, x2, x2], [y, y + height, y + height, y], color='black', linewidth=0.9)
    ax.text((x1 + x2) / 2, y + height + 0.008, bias_significance_label(p),
            ha='center', va='bottom', fontsize=8)

def plot_bias_pair_panel(ax, components, mode, title):
    offset = 0.13
    for x, (condition, level) in enumerate(BIAS_CONDITIONS):
        selected = bias_condition_components(components, condition, level)
        nm, ne, bm, be, p, session_n = bias_pair_stats(selected, mode)
        ax.errorbar(x - offset, nm, yerr=ne, fmt='o', color=NEUTRAL_COLOR,
                    markersize=6, capsize=4, elinewidth=1.4)
        ax.errorbar(x + offset, bm, yerr=be, fmt='o', color=BLOCK_COLOR,
                    markersize=6, capsize=4, elinewidth=1.4)
        if np.isfinite(nm) and np.isfinite(bm):
            add_bias_pair_bracket(ax, x - offset, x + offset, max(nm + ne, bm + be), p)
        ax.text(x, -0.18, f'{session_n} sess', ha='center', va='top', fontsize=7,
                transform=ax.get_xaxis_transform(), clip_on=False)
    ax.set_xlim(-0.5, 3.5)
    ax.margins(y=0.18)
    ax.set_xticks(range(4), BIAS_LABELS)
    ax.set_ylabel('P(Choose Long-Associated Response)')
    ax.set_title(title, fontsize=11)
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(handles=[
        Line2D([0], [0], marker='o', linestyle='none', color=NEUTRAL_COLOR, label='Matched neutral'),
        Line2D([0], [0], marker='o', linestyle='none', color=BLOCK_COLOR, label='Block'),
    ], frameon=False, fontsize=8, loc='best')

def bias_context_subset(components, context):
    if context is None:
        return components
    return components[components['block_context'].eq(context)].copy()

def build_bias_page(components, analysis_label='Exact session ISI extremes'):
    fig = plt.figure(figsize=(48, 26))
    gs = gridspec.GridSpec(5, 6, figure=fig, hspace=0.58, wspace=0.38)
    for row, (label, subject) in enumerate(SUBJECT_ROWS):
        row_components = (
            components if subject is None else components[components['subject'].eq(subject)].copy()
        )
        plot_bias_panel(fig.add_subplot(gs[row, 0]), row_components, 'pooled',
                        f'{label} — pooled — all blocks')
        plot_bias_panel(fig.add_subplot(gs[row, 1]),
                        bias_context_subset(row_components, 'Short block'), 'pooled',
                        f'{label} — pooled — short block')
        plot_bias_panel(fig.add_subplot(gs[row, 2]),
                        bias_context_subset(row_components, 'Long block'), 'pooled',
                        f'{label} — pooled — long block')
        plot_bias_panel(fig.add_subplot(gs[row, 3]), row_components, 'grand',
                        f'{label} — grand average — all blocks')
        plot_bias_panel(fig.add_subplot(gs[row, 4]),
                        bias_context_subset(row_components, 'Short block'), 'grand',
                        f'{label} — grand average — short block')
        plot_bias_panel(fig.add_subplot(gs[row, 5]),
                        bias_context_subset(row_components, 'Long block'), 'grand',
                        f'{label} — grand average — long block')
    fig.suptitle(
        f'Common/rare choice bias relative to non-naive neutral trials — {analysis_label}',
        fontsize=16, y=0.995
    )
    fig.subplots_adjust(left=0.08, right=0.97, top=0.975, bottom=0.055)
    return fig

def build_bias_pair_page(components, analysis_label='Exact session ISI extremes'):
    fig = plt.figure(figsize=(48, 26))
    gs = gridspec.GridSpec(5, 6, figure=fig, hspace=0.58, wspace=0.38)
    for row, (label, subject) in enumerate(SUBJECT_ROWS):
        row_components = (
            components if subject is None else components[components['subject'].eq(subject)].copy()
        )
        plot_bias_pair_panel(fig.add_subplot(gs[row, 0]), row_components, 'pooled',
                             f'{label} — pooled — all blocks')
        plot_bias_pair_panel(fig.add_subplot(gs[row, 1]),
                             bias_context_subset(row_components, 'Short block'), 'pooled',
                             f'{label} — pooled — short block')
        plot_bias_pair_panel(fig.add_subplot(gs[row, 2]),
                             bias_context_subset(row_components, 'Long block'), 'pooled',
                             f'{label} — pooled — long block')
        plot_bias_pair_panel(fig.add_subplot(gs[row, 3]), row_components, 'grand',
                             f'{label} — grand average — all blocks')
        plot_bias_pair_panel(fig.add_subplot(gs[row, 4]),
                             bias_context_subset(row_components, 'Short block'), 'grand',
                             f'{label} — grand average — short block')
        plot_bias_pair_panel(fig.add_subplot(gs[row, 5]),
                             bias_context_subset(row_components, 'Long block'), 'grand',
                             f'{label} — grand average — long block')
    fig.suptitle(
        f'Block versus matched non-naive neutral P(Choose Long-Associated Response) — {analysis_label}\n'
        'Pooled: Fisher exact; grand average: paired Wilcoxon', fontsize=16, y=0.995
    )
    fig.subplots_adjust(left=0.08, right=0.97, top=0.965, bottom=0.055)
    return fig

def plot_bias_distribution_panel(ax, components, mode, title):
    rng = np.random.default_rng(12345)
    for x, ((condition, level), color) in enumerate(zip(BIAS_CONDITIONS, BIAS_COLORS)):
        selected = bias_condition_components(components, condition, level)
        stats = bias_zero_test(selected, mode)
        values = stats['values'].to_numpy(float)
        if len(values) < 2:
            continue
        violin = ax.violinplot(values, positions=[x], widths=0.72,
                               showmeans=False, showmedians=False, showextrema=False)
        for body in violin['bodies']:
            body.set_facecolor(color); body.set_edgecolor(color); body.set_alpha(0.28)
            for path in body.get_paths():
                path.vertices[:, 0] = np.maximum(path.vertices[:, 0], x)
        shown = values if len(values) <= 350 else rng.choice(values, 350, replace=False)
        jitter = x + rng.uniform(0.025, 0.27, len(shown))
        ax.scatter(jitter, shown, s=4, color=color, alpha=0.22, linewidths=0)
        q1, median, q3 = np.quantile(values, [0.25, 0.5, 0.75])
        ax.plot([x, x], [q1, q3], color='black', linewidth=2.2, zorder=5)
        ax.plot([x - 0.025, x + 0.14], [median, median], color='black', linewidth=1.4, zorder=6)
        ax.scatter([x], [median], marker='D', s=28, color=color,
                   edgecolor='white', linewidth=0.5, zorder=7)
        span = max(float(values.max() - values.min()), 0.05)
        ax.text(x + 0.13, float(values.max()) + 0.06 * span,
                bias_significance_label(stats['p']), ha='center', va='bottom',
                fontsize=8, fontweight='bold')
        ax.text(x, -0.18, f'n={len(values)}\n{stats["test"]}', ha='center', va='top',
                fontsize=6, transform=ax.get_xaxis_transform(), clip_on=False)
    ax.axhline(0, color='0.35', linestyle=':', linewidth=1)
    ax.set_xlim(-0.5, 3.65); ax.margins(y=0.14)
    ax.set_xticks(range(4), BIAS_LABELS)
    ax.set_ylabel('Bias distribution\nΔP(Long-associated response)')
    ax.set_title(title, fontsize=10)
    ax.grid(False); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

def build_bias_distribution_page(components, analysis_label='Exact session ISI extremes'):
    fig = plt.figure(figsize=(48, 26))
    gs = gridspec.GridSpec(5, 6, figure=fig, hspace=0.62, wspace=0.38)
    for row, (label, subject) in enumerate(SUBJECT_ROWS):
        row_components = components if subject is None else components[components['subject'].eq(subject)].copy()
        panel_specs = [
            (row_components, 'pooled', f'{label} — pooled — all blocks'),
            (bias_context_subset(row_components, 'Short block'), 'pooled', f'{label} — pooled — short block'),
            (bias_context_subset(row_components, 'Long block'), 'pooled', f'{label} — pooled — long block'),
            (row_components, 'grand', f'{label} — grand average — all blocks'),
            (bias_context_subset(row_components, 'Short block'), 'grand', f'{label} — grand average — short block'),
            (bias_context_subset(row_components, 'Long block'), 'grand', f'{label} — grand average — long block'),
        ]
        for column, (panel_data, mode, title) in enumerate(panel_specs):
            plot_bias_distribution_panel(fig.add_subplot(gs[row, column]), panel_data, mode, title)
    fig.suptitle(f'Bias distributions — {analysis_label}: median + IQR and two-sided tests versus zero\n'
                 'All visual summaries use median + IQR; Shapiro p < 0.01 selects Wilcoxon, otherwise one-sample t-test',
                 fontsize=16, y=0.995)
    fig.subplots_adjust(left=0.045, right=0.985, top=0.965, bottom=0.065)
    return fig

def summarize_bias_bin_curve(components, mode):
    rows = []
    if components.empty:
        return pd.DataFrame(rows)
    group_columns = ['condition', 'trial_type', 'bin_index']
    for keys, group in components.groupby(group_columns, sort=True):
        condition, trial_type, bin_index = keys
        if mode == 'pooled':
            block_n = int(group['block_n'].sum())
            neutral_n = int(group['neutral_n'].sum())
            block_p = float(group['block_success'].sum() / block_n)
            neutral_p = float(group['neutral_success'].sum() / neutral_n)
            mean = block_p - neutral_p
            sem = float(np.sqrt(
                block_p * (1 - block_p) / block_n
                + neutral_p * (1 - neutral_p) / neutral_n
            ))
            isi_weights = group['block_n'] + group['neutral_n']
            isi_mean = float(np.average(group['isi_mean'], weights=isi_weights))
            n = block_n
        elif mode == 'grand':
            session_values = group.groupby('session').agg(
                bias=('bias', 'mean'), isi_mean=('isi_mean', 'mean')
            )
            mean = float(session_values['bias'].mean())
            sem = float(session_values['bias'].sem()) if len(session_values) > 1 else 0.0
            isi_mean = float(session_values['isi_mean'].mean())
            n = len(session_values)
        else:
            raise ValueError("mode must be 'pooled' or 'grand'")
        rows.append({
            'condition': condition, 'trial_type': int(trial_type),
            'bin_index': int(bin_index), 'isi_mean': isi_mean,
            'mean': mean, 'sem': sem, 'n': n,
        })
    return pd.DataFrame(rows)

def plot_bias_bin_curve_panel(ax, components, mode, title):
    summary = summarize_bias_bin_curve(components, mode)
    curve_colors = {'Common': '#1b9e77', 'Rare': '#d95f02'}
    trial_markers = {1: 'o', 2: 's'}
    trial_labels = {1: 'Short ISI', 2: 'Long ISI'}
    for condition in ['Common', 'Rare']:
        for trial_type in [1, 2]:
            curve = summary[
                summary['condition'].eq(condition)
                & summary['trial_type'].eq(trial_type)
            ].sort_values('bin_index') if not summary.empty else pd.DataFrame()
            if curve.empty:
                continue
            ax.errorbar(
                curve['isi_mean'], curve['mean'], yerr=curve['sem'],
                marker=trial_markers[trial_type], markersize=4.5, capsize=2.5,
                linewidth=1.5, color=curve_colors[condition],
                label=f'{condition} — {trial_labels[trial_type]}',
            )
    ax.axhline(0, color='0.45', linestyle=':', linewidth=1)
    ax.axvline(1.25, color='0.55', linestyle='--', linewidth=0.9, alpha=0.6)
    ax.set_xlim(0.1, 2.4)
    ax.set_xlabel('ISI (s)')
    ax.set_ylabel('Choice bias: ΔP(Long-associated response)\n(block − matched neutral)')
    ax.set_title(title, fontsize=10)
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles, labels, frameon=False, fontsize=6, loc='best')

def build_bias_bin_curve_page(components):
    fig = plt.figure(figsize=(48, 26))
    gs = gridspec.GridSpec(5, 6, figure=fig, hspace=0.62, wspace=0.38)
    for row, (label, subject) in enumerate(SUBJECT_ROWS):
        row_components = (
            components if subject is None else components[components['subject'].eq(subject)].copy()
        )
        panel_specs = [
            (row_components, 'pooled', f'{label} — pooled — all blocks'),
            (bias_context_subset(row_components, 'Short block'), 'pooled', f'{label} — pooled — short block'),
            (bias_context_subset(row_components, 'Long block'), 'pooled', f'{label} — pooled — long block'),
            (row_components, 'grand', f'{label} — grand average — all blocks'),
            (bias_context_subset(row_components, 'Short block'), 'grand', f'{label} — grand average — short block'),
            (bias_context_subset(row_components, 'Long block'), 'grand', f'{label} — grand average — long block'),
        ]
        for column, (panel_data, mode, title) in enumerate(panel_specs):
            plot_bias_bin_curve_panel(fig.add_subplot(gs[row, column]), panel_data, mode, title)
    fig.suptitle(
        'Bin-resolved psychometric choice bias: block minus matched non-naive neutral',
        fontsize=16, y=0.995
    )
    fig.subplots_adjust(left=0.06, right=0.985, top=0.97, bottom=0.055)
    return fig

def collect_bias_zero_test_table(components):
    rows = []
    for row_label, subject in SUBJECT_ROWS:
        row_components = components if subject is None else components[components['subject'].eq(subject)]
        for context in [None, 'Short block', 'Long block']:
            panel_data = bias_context_subset(row_components, context)
            for mode in ['pooled', 'grand']:
                for condition, level in BIAS_CONDITIONS:
                    stats = bias_zero_test(bias_condition_components(panel_data, condition, level), mode)
                    rows.append({'row': row_label, 'context': context or 'All blocks', 'mode': mode,
                                 'condition': condition, 'difficulty': level, 'n': len(stats['values']),
                                 'summary': 'median + IQR' if stats['non_normal'] else 'mean ± SEM',
                                 'normality_p': stats['normality_p'], 'test': stats['test'],
                                 'center': stats['center'], 'p_vs_zero': stats['p']})
    return pd.DataFrame(rows)

def make_bias_figure(components, binned_components, bin_curve_components, output_path=BIAS_OUTPUT_PATH):
    bias_page = build_bias_page(components)
    pair_page = build_bias_pair_page(components)
    distribution_page = build_bias_distribution_page(components)
    bin_label = 'Four psychometric bins (outer bins define easy/hard)'
    binned_bias_page = build_bias_page(binned_components, bin_label)
    binned_pair_page = build_bias_pair_page(binned_components, bin_label)
    binned_distribution_page = build_bias_distribution_page(binned_components, bin_label)
    bin_curve_page = build_bias_bin_curve_page(bin_curve_components)
    output_path = Path(output_path)
    with PdfPages(output_path) as pdf:
        pdf.savefig(bias_page, bbox_inches='tight')
        pdf.savefig(pair_page, bbox_inches='tight')
        pdf.savefig(distribution_page, bbox_inches='tight')
        pdf.savefig(binned_bias_page, bbox_inches='tight')
        pdf.savefig(binned_pair_page, bbox_inches='tight')
        pdf.savefig(binned_distribution_page, bbox_inches='tight')
        pdf.savefig(bin_curve_page, bbox_inches='tight')
    plt.show()
    return output_path

bias_data = bias_prepare_data()
bias_components = collect_bias_components(bias_data)
binned_bias_components = collect_binned_bias_components(bias_data, n_bins_per_type=4)
bias_bin_curve_components = collect_bias_bin_curve_components(bias_data, n_bins_per_type=4)
bias_zero_test_table = collect_bias_zero_test_table(bias_components)
binned_bias_zero_test_table = collect_bias_zero_test_table(binned_bias_components)
bias_pdf_path = make_bias_figure(
    bias_components, binned_bias_components, bias_bin_curve_components
)
print(f'Saved: {bias_pdf_path.resolve()}')
print(bias_components.groupby(['subject', 'condition', 'difficulty']).agg(
    sessions=('session', 'nunique'), block_trials=('block_n', 'sum'),
    neutral_trials=('neutral_n', 'sum'), mean_session_component_bias=('bias', 'mean'),
))
print('\nTests versus zero:')
print(bias_zero_test_table.to_string(index=False))
print('\nBinned tests versus zero:')
print(binned_bias_zero_test_table.to_string(index=False))


Saved: /home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/choice_bias_relative_to_neutral.pdf
                              sessions  block_trials  neutral_trials  \
subject condition difficulty                                           
MC06    Common    Difficult          1            30               6   
                  Easy               1            43               5   
        Rare      Difficult          1             7               6   
                  Easy               1            10               5   

                              mean_session_component_bias  
subject condition difficulty                               
MC06    Common    Difficult                     -0.208899  
                  Easy                           0.021739  
        Rare      Difficult                     -0.250000  
                  Easy                           0.000000  

Tests versus zero:
         row     context   m

In [6]:
# Distribution of detected and transition-aligned block lengths
BLOCK_LENGTH_OUTPUT_PATH = Path('block_length_distributions.pdf')
BLOCK_LENGTH_COLORS = {1: '#BCAE3E', 2: 'blue'}
BLOCK_LENGTH_LABELS = {1: 'Short block', 2: 'Long block'}

def collect_block_lengths(data):
    records = []
    for session_key, session_df in data.groupby('change_session', sort=False):
        # Use the same ordered short/long trial stream as the block-aligned analysis.
        sdf = session_df[session_df['block_type'].isin([1, 2])].copy()
        sdf = sdf.sort_values('_source_order').reset_index(drop=True)
        if sdf.empty:
            continue

        formal_switches = sdf.index[
            sdf['block_type'].ne(sdf['block_type'].shift())
            & sdf['block_type'].shift().notna()
        ].tolist()

        # Detected blocks are contiguous runs of the recorded block_type.
        # Pairing each start only with the next start omits the session's final block.
        detected_starts = [0] + formal_switches
        for block_number, (start, stop) in enumerate(zip(detected_starts[:-1], detected_starts[1:])):
            block_type = int(sdf.loc[start, 'block_type'])
            records.append({
                'subject': sdf.loc[start, 'subject'],
                'session': session_key,
                'block_number': block_number,
                'block_type': block_type,
                'block_label': BLOCK_LENGTH_LABELS[block_type],
                'definition': 'Detected block length',
                'block_length': int(stop - start),
            })

        # Reproduce the transition-alignment rule exactly: move each switch backward
        # through the terminal rare-trial run until the preceding trial is non-rare.
        adjusted_boundaries = [(0, int(sdf.loc[0, 'block_type']))]
        for formal_switch in formal_switches:
            adjusted_start = adjusted_block_start(sdf, formal_switch)
            if adjusted_start not in {boundary[0] for boundary in adjusted_boundaries}:
                adjusted_boundaries.append((
                    adjusted_start, int(sdf.loc[formal_switch, 'block_type'])
                ))
        # As above, the unpaired final start means the final session block is excluded.
        for block_number, ((start, block_type), (stop, _)) in enumerate(
            zip(adjusted_boundaries[:-1], adjusted_boundaries[1:])
        ):
            records.append({
                'subject': sdf.loc[start, 'subject'],
                'session': session_key,
                'block_number': block_number,
                'block_type': block_type,
                'block_label': BLOCK_LENGTH_LABELS[block_type],
                'definition': 'Transition-aligned block length',
                'block_length': int(stop - start),
            })
    return pd.DataFrame(records)

def plot_block_length_distributions(lengths, output_path=BLOCK_LENGTH_OUTPUT_PATH):
    definitions = ['Detected block length', 'Transition-aligned block length']
    rows = [
        ('All subjects, all sessions', None),
        ('YH24LG', 'YH24LG'),
        ('MC06', 'MC06'),
        ('MC07', 'MC07'),
        ('MC08', 'MC08'),
    ]
    minimum = int(lengths['block_length'].min())
    maximum = int(lengths['block_length'].max())
    bins = np.arange(minimum - 0.5, maximum + 1.5, 1)
    fig, axes = plt.subplots(len(rows), 2, figsize=(16, 23), sharex=True, sharey=False)

    for row_index, (row_label, subject) in enumerate(rows):
        row_data = lengths if subject is None else lengths[lengths['subject'].eq(subject)]
        for column_index, definition in enumerate(definitions):
            ax = axes[row_index, column_index]
            panel = row_data[row_data['definition'].eq(definition)]
            panel_y_max = 0.0
            for block_type in [1, 2]:
                values = panel.loc[panel['block_type'].eq(block_type), 'block_length'].dropna()
                weights = np.full(len(values), 100.0 / len(values)) if len(values) else None
                if len(values):
                    histogram_percent, _ = np.histogram(values, bins=bins, weights=weights)
                    panel_y_max = max(panel_y_max, float(histogram_percent.max()))
                color = BLOCK_LENGTH_COLORS[block_type]
                mean = float(values.mean()) if len(values) else np.nan
                sem = float(values.sem()) if len(values) > 1 else (0.0 if len(values) else np.nan)
                median = float(values.median()) if len(values) else np.nan
                ax.hist(values, bins=bins, weights=weights, histtype='step', linewidth=2.2,
                        color=color,
                        label=(f'{BLOCK_LENGTH_LABELS[block_type]} (n={len(values)}, '
                               f'mean={mean:.2f} ± {sem:.2f} SEM, median={median:g})'))
                if np.isfinite(median):
                    ax.axvline(median, color=color, linestyle='--', linewidth=1.3, alpha=0.9)
                if np.isfinite(mean) and np.isfinite(sem):
                    # Horizontal mean ± SEM bars at the top of each distribution panel.
                    top_y = {1: 0.96, 2: 0.89}[block_type]
                    ax.errorbar(mean, top_y, xerr=sem, fmt='o', color=color,
                                markersize=5.5, capsize=5, capthick=1.6,
                                elinewidth=2.2, transform=ax.get_xaxis_transform(),
                                clip_on=False, zorder=5)
            if row_index == 0:
                ax.set_title(definition, fontsize=13)
            if column_index == 0:
                ax.set_ylabel(f'{row_label}\nBlocks at each length (%)')
            if row_index == len(rows) - 1:
                ax.set_xlabel('Block length (trials)')
            ax.set_xlim(minimum - 0.5, 35)
            ax.set_ylim(0, panel_y_max * 1.10 if panel_y_max > 0 else 1)
            ax.grid(axis='y', alpha=0.2)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.legend(frameon=False, fontsize=8, loc='upper right')
    fig.suptitle(
        'Block-length distributions (neutral blocks and each session’s final block excluded)',
        fontsize=15, y=0.995
    )
    fig.text(0.5, 0.008,
             'Transition-aligned boundary: move backward over consecutive rare trials immediately before each recorded switch.',
             ha='center', fontsize=9)
    fig.tight_layout(rect=[0, 0.025, 1, 0.98], h_pad=2.2, w_pad=1.5)
    output_path = Path(output_path)
    fig.savefig(output_path, bbox_inches='tight')
    plt.show()
    return output_path

block_length_table = collect_block_lengths(block_change_data)
block_length_pdf_path = plot_block_length_distributions(block_length_table)
print(f'Saved: {block_length_pdf_path.resolve()}')
print(block_length_table.groupby(['definition', 'block_label'])['block_length'].describe())


Saved: /home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/block_length_distributions.pdf
                                             count   mean       std   min  \
definition                      block_label                                 
Detected block length           Long block     4.0  24.25  0.957427  23.0   
                                Short block    5.0  24.00  1.581139  22.0   
Transition-aligned block length Long block     4.0  23.50  0.577350  23.0   
                                Short block    5.0  24.60  1.516575  22.0   

                                               25%   50%   75%   max  
definition                      block_label                           
Detected block length           Long block   23.75  24.5  25.0  25.0  
                                Short block  23.00  24.0  25.0  26.0  
Transition-aligned block length Long block   23.00  23.5  24.0  24.0  
                            

In [7]:
# Page 1: context benefit and context cost across four-bin ISI psychometrics
CONTEXT_EFFECT_OUTPUT_PATH = Path('context_benefit_cost_by_isi_bin.pdf')
import gc
CONTEXT_EFFECT_ROWS = [
    ('All subjects, all sessions', None),
    ('YH24LG', 'YH24LG'),
    ('MC06', 'MC06'),
    ('MC07', 'MC07'),
    ('MC08', 'MC08'),
]
CONTEXT_EFFECT_COLORS = {'Benefit': '#1b9e77', 'Cost': '#d95f02'}
CONTEXT_EFFECT_MARKERS = {'Benefit': 'o', 'Cost': 's'}
CONTEXT_SUMMARY_METHOD_COLORS = {'Bin first': '#0072B2', 'Effect first': '#CC79A7'}
from scipy.stats import norm, mannwhitneyu
SEQUENCE_EASY_COLOR = '#1b9e77'
SEQUENCE_HARD_COLOR = '#d95f02'
SEQUENCE_PHASE_MARKERS = {'pre': 'o', 'rare': 's', 'post': '^'}

def collect_context_effect_components(data, n_bins_per_type=4):
    """Build session-level common-neutral and rare-neutral correctness contrasts."""
    binned = bias_add_psychometric_bins(data, n_bins_per_type=n_bins_per_type)
    binned['correct'] = binned['outcome'].isin(CORRECT_OUTCOMES).astype(float)
    rows = []
    for session, session_data in binned.groupby('bias_session', sort=False):
        subject = session_data['subject'].iloc[0]
        for trial_type in [1, 2]:
            bin_values = sorted(
                session_data.loc[
                    session_data['trial_type'].eq(trial_type), 'psychometric_bin'
                ].dropna().unique()
            )
            for bin_index in bin_values:
                same_stimulus_bin = (
                    session_data['trial_type'].eq(trial_type)
                    & session_data['psychometric_bin'].eq(bin_index)
                )
                neutral = session_data.loc[
                    same_stimulus_bin & session_data['block_type'].eq(0)
                ]
                common = session_data.loc[
                    same_stimulus_bin & session_data['block_type'].eq(trial_type)
                ]
                rare = session_data.loc[
                    same_stimulus_bin & session_data['block_type'].eq(3 - trial_type)
                ]

                comparisons = [
                    ('Benefit', common, 'Short block' if trial_type == 1 else 'Long block'),
                    ('Cost', rare, 'Long block' if trial_type == 1 else 'Short block'),
                ]
                for contrast, condition_trials, block_context in comparisons:
                    if neutral.empty or condition_trials.empty:
                        continue
                    condition_p = float(condition_trials['correct'].mean())
                    neutral_p = float(neutral['correct'].mean())
                    effect = (
                        condition_p - neutral_p
                        if contrast == 'Benefit'
                        else condition_p - neutral_p
                    )
                    matched_isi = pd.concat(
                        [condition_trials['trial_isi'], neutral['trial_isi']],
                        ignore_index=True,
                    )
                    rows.append({
                        'subject': subject,
                        'session': session,
                        'contrast': contrast,
                        'block_context': block_context,
                        'trial_type': int(trial_type),
                        'bin_index': int(bin_index),
                        'isi_mean': float(matched_isi.mean()),
                        'effect': effect,
                        'condition_success': int(condition_trials['correct'].sum()),
                        'condition_n': int(len(condition_trials)),
                        'neutral_success': int(neutral['correct'].sum()),
                        'neutral_n': int(len(neutral)),
                    })
    return pd.DataFrame(rows)

def collect_exact_isi_context_effect_components(data):
    """Calculate benefit/cost at each exact ISI before any binning."""
    exact_data = data.copy()
    exact_data['correct'] = exact_data['outcome'].isin(CORRECT_OUTCOMES).astype(float)
    rows = []
    for session, session_data in exact_data.groupby('bias_session', sort=False):
        subject = session_data['subject'].iloc[0]
        for trial_type in [1, 2]:
            trial_type_data = session_data[session_data['trial_type'].eq(trial_type)]
            for exact_isi in sorted(trial_type_data['trial_isi'].dropna().unique()):
                same_stimulus = (
                    session_data['trial_type'].eq(trial_type)
                    & session_data['trial_isi'].eq(exact_isi)
                )
                neutral = session_data.loc[
                    same_stimulus & session_data['block_type'].eq(0)
                ]
                common = session_data.loc[
                    same_stimulus & session_data['block_type'].eq(trial_type)
                ]
                rare = session_data.loc[
                    same_stimulus & session_data['block_type'].eq(3 - trial_type)
                ]
                comparisons = [
                    ('Benefit', common, 'Short block' if trial_type == 1 else 'Long block'),
                    ('Cost', rare, 'Long block' if trial_type == 1 else 'Short block'),
                ]
                for contrast, condition_trials, block_context in comparisons:
                    if neutral.empty or condition_trials.empty:
                        continue
                    condition_p = float(condition_trials['correct'].mean())
                    neutral_p = float(neutral['correct'].mean())
                    effect = condition_p - neutral_p
                    rows.append({
                        'subject': subject, 'session': session, 'contrast': contrast,
                        'block_context': block_context, 'trial_type': int(trial_type),
                        'exact_isi': float(exact_isi), 'effect': effect,
                        'condition_success': int(condition_trials['correct'].sum()),
                        'condition_n': int(len(condition_trials)),
                        'neutral_success': int(neutral['correct'].sum()),
                        'neutral_n': int(len(neutral)),
                    })
    return pd.DataFrame(rows)

def bin_precomputed_context_effects(components, n_bins_per_type=4):
    """Assign already-calculated exact-ISI contrasts to four ISI bins."""
    binned = components.copy()
    binned['bin_index'] = np.nan
    for (_, trial_type), group in binned.groupby(['session', 'trial_type'], sort=False):
        exact_values = np.array(sorted(group['exact_isi'].dropna().unique()), dtype=float)
        if len(exact_values) == 0:
            continue
        np.random.seed(42)
        jittered_values = pd.Series(
            exact_values + np.random.normal(0, 1e-6, len(exact_values))
        )
        try:
            bin_codes = pd.qcut(
                jittered_values, q=n_bins_per_type, labels=False, duplicates='drop'
            )
        except ValueError:
            continue
        isi_to_bin = dict(zip(exact_values, bin_codes.to_numpy(dtype=float)))
        for exact_isi, bin_index in isi_to_bin.items():
            mask = (
                binned.index.isin(group.index)
                & binned['exact_isi'].eq(exact_isi)
            )
            binned.loc[mask, 'bin_index'] = bin_index
    return binned.dropna(subset=['bin_index']).assign(
        bin_index=lambda frame: frame['bin_index'].astype(int)
    )

def summarize_precomputed_context_effects(components, mode):
    """Aggregate contrasts that were calculated before bin assignment."""
    rows = []
    if components.empty:
        return pd.DataFrame(rows)
    for (contrast, trial_type, bin_index), group in components.groupby(
        ['contrast', 'trial_type', 'bin_index'], sort=True
    ):
        if mode == 'pooled':
            weights = (group['condition_n'] + group['neutral_n']).to_numpy(dtype=float)
            normalized_weights = weights / weights.sum()
            effect_values = group['effect'].to_numpy(dtype=float)
            mean = float(np.sum(normalized_weights * effect_values))
            weight_square_sum = float(np.sum(normalized_weights ** 2))
            if len(effect_values) > 1 and weight_square_sum < 1:
                weighted_variance = float(
                    np.sum(normalized_weights * (effect_values - mean) ** 2)
                    / (1 - weight_square_sum)
                )
                sem = float(np.sqrt(weighted_variance * weight_square_sum))
            else:
                sem = 0.0
            isi_mean = float(np.average(group['exact_isi'], weights=weights))
            n = int(group['condition_n'].sum())
        elif mode == 'grand':
            session_values = group.groupby('session').agg(
                effect=('effect', 'mean'), exact_isi=('exact_isi', 'mean')
            )
            mean = float(session_values['effect'].mean())
            sem = float(session_values['effect'].sem()) if len(session_values) > 1 else 0.0
            isi_mean = float(session_values['exact_isi'].mean())
            n = int(len(session_values))
        else:
            raise ValueError("mode must be 'pooled' or 'grand'")
        rows.append({
            'contrast': contrast, 'trial_type': int(trial_type),
            'bin_index': int(bin_index), 'isi_mean': isi_mean,
            'mean': mean, 'sem': sem, 'n': n,
        })
    return pd.DataFrame(rows)

def summarize_context_effects(components, mode):
    rows = []
    if components.empty:
        return pd.DataFrame(rows)
    group_columns = ['contrast', 'trial_type', 'bin_index']
    for (contrast, trial_type, bin_index), group in components.groupby(
        group_columns, sort=True
    ):
        if mode == 'pooled':
            condition_n = int(group['condition_n'].sum())
            neutral_n = int(group['neutral_n'].sum())
            condition_p = float(group['condition_success'].sum() / condition_n)
            neutral_p = float(group['neutral_success'].sum() / neutral_n)
            mean = (
                condition_p - neutral_p
                if contrast == 'Benefit'
                else condition_p - neutral_p
            )
            sem = float(np.sqrt(
                condition_p * (1 - condition_p) / condition_n
                + neutral_p * (1 - neutral_p) / neutral_n
            ))
            isi_weights = group['condition_n'] + group['neutral_n']
            isi_mean = float(np.average(group['isi_mean'], weights=isi_weights))
            n = condition_n
        elif mode == 'grand':
            session_values = group.groupby('session').agg(
                effect=('effect', 'mean'),
                isi_mean=('isi_mean', 'mean'),
            )
            mean = float(session_values['effect'].mean())
            sem = (
                float(session_values['effect'].sem())
                if len(session_values) > 1 else 0.0
            )
            isi_mean = float(session_values['isi_mean'].mean())
            n = int(len(session_values))
        else:
            raise ValueError("mode must be 'pooled' or 'grand'")
        rows.append({
            'contrast': contrast,
            'trial_type': int(trial_type),
            'bin_index': int(bin_index),
            'isi_mean': isi_mean,
            'mean': mean,
            'sem': sem,
            'n': n,
        })
    return pd.DataFrame(rows)

def context_effect_subset(components, block_context):
    if block_context is None:
        return components
    return components[components['block_context'].eq(block_context)].copy()

def plot_context_effect_panel(
    ax, components, mode, title, summary_function=summarize_context_effects
):
    summary = summary_function(components, mode)
    for contrast in ['Benefit', 'Cost']:
        curve = (
            summary[summary['contrast'].eq(contrast)].sort_values('isi_mean')
            if not summary.empty else pd.DataFrame()
        )
        if curve.empty:
            continue
        ax.errorbar(
            curve['isi_mean'], curve['mean'], yerr=curve['sem'],
            marker=CONTEXT_EFFECT_MARKERS[contrast], markersize=4.5,
            capsize=2.5, linewidth=1.5, color=CONTEXT_EFFECT_COLORS[contrast],
            label=(
                'Benefit: P(correct | common) − P(correct | neutral)'
                if contrast == 'Benefit'
                else 'Cost: P(correct | rare) − P(correct | neutral)'
            ),
        )
    ax.axhline(0, color='0.45', linestyle=':', linewidth=1)
    ax.axvline(1.25, color='0.55', linestyle='--', linewidth=0.9, alpha=0.6)
    ax.margins(x=0.04, y=0.18)
    ax.set_xlabel('ISI-bin mean (s)')
    ax.set_ylabel('Probability difference (mean ± SEM)')
    ax.set_title(title, fontsize=10)
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(frameon=False, fontsize=6, loc='best')

def build_context_effect_page(components):
    fig = plt.figure(figsize=(48, 26))
    gs = gridspec.GridSpec(5, 6, figure=fig, hspace=0.64, wspace=0.38)
    for row, (row_label, subject) in enumerate(CONTEXT_EFFECT_ROWS):
        row_components = (
            components
            if subject is None
            else components[components['subject'].eq(subject)].copy()
        )
        panel_specs = [
            (row_components, 'pooled', f'{row_label} — pooled — all'),
            (context_effect_subset(row_components, 'Short block'), 'pooled',
             f'{row_label} — pooled — short block'),
            (context_effect_subset(row_components, 'Long block'), 'pooled',
             f'{row_label} — pooled — long block'),
            (row_components, 'grand', f'{row_label} — grand average — all'),
            (context_effect_subset(row_components, 'Short block'), 'grand',
             f'{row_label} — grand average — short block'),
            (context_effect_subset(row_components, 'Long block'), 'grand',
             f'{row_label} — grand average — long block'),
        ]
        for column, (panel_data, mode, title) in enumerate(panel_specs):
            plot_context_effect_panel(
                fig.add_subplot(gs[row, column]), panel_data, mode, title
            )
    fig.suptitle(
        'Context benefit and cost across four-bin ISI psychometrics\n'
        'Benefit = P(correct | common) − P(correct | neutral); Cost = P(correct | rare) − P(correct | neutral)\n'
        'Pooled trials versus equal-session-weighted grand averages',
        fontsize=16, y=0.995,
    )
    fig.subplots_adjust(left=0.055, right=0.99, top=0.965, bottom=0.055)
    return fig

def build_precomputed_then_binned_context_effect_page(components):
    fig = plt.figure(figsize=(48, 26))
    gs = gridspec.GridSpec(5, 6, figure=fig, hspace=0.64, wspace=0.38)
    for row, (row_label, subject) in enumerate(CONTEXT_EFFECT_ROWS):
        row_components = (
            components
            if subject is None
            else components[components['subject'].eq(subject)].copy()
        )
        panel_specs = [
            (row_components, 'pooled', f'{row_label} — pooled — all'),
            (context_effect_subset(row_components, 'Short block'), 'pooled',
             f'{row_label} — pooled — short block'),
            (context_effect_subset(row_components, 'Long block'), 'pooled',
             f'{row_label} — pooled — long block'),
            (row_components, 'grand', f'{row_label} — grand average — all'),
            (context_effect_subset(row_components, 'Short block'), 'grand',
             f'{row_label} — grand average — short block'),
            (context_effect_subset(row_components, 'Long block'), 'grand',
             f'{row_label} — grand average — long block'),
        ]
        for column, (panel_data, mode, title) in enumerate(panel_specs):
            plot_context_effect_panel(
                fig.add_subplot(gs[row, column]), panel_data, mode, title,
                summary_function=summarize_precomputed_context_effects,
            )
    fig.suptitle(
        'ORDER: benefit/cost calculated at each exact ISI FIRST, then the calculated contrasts are grouped into four ISI bins\n'
        'Benefit = P(correct | common) − P(correct | neutral); Cost = P(correct | rare) − P(correct | neutral)\n'
        'Pooled: trial-count-weighted exact-ISI contrasts; grand average: equal-session-weighted contrasts',
        fontsize=16, y=0.995,
    )
    fig.subplots_adjust(left=0.055, right=0.99, top=0.955, bottom=0.055)
    return fig

def add_context_effect_difficulty(components):
    """Keep the outer bins and label easy/hard according to stimulus class."""
    labeled = components.copy()
    labeled['difficulty'] = pd.NA
    for (_, trial_type), group in labeled.groupby(['session', 'trial_type'], sort=False):
        bin_values = sorted(group['bin_index'].dropna().unique())
        if len(bin_values) < 2:
            continue
        lowest_bin, highest_bin = bin_values[0], bin_values[-1]
        easy_bin, hard_bin = (
            (lowest_bin, highest_bin)
            if int(trial_type) == 1 else (highest_bin, lowest_bin)
        )
        labeled.loc[group.index[group['bin_index'].eq(easy_bin)], 'difficulty'] = 'Easy'
        labeled.loc[group.index[group['bin_index'].eq(hard_bin)], 'difficulty'] = 'Hard'
    return labeled.dropna(subset=['difficulty']).copy()

def context_summary_category_stats(
    components, contrast, difficulty, mode, precomputed
):
    selected = components[
        components['contrast'].eq(contrast)
        & components['difficulty'].eq(difficulty)
    ].copy()
    if selected.empty:
        return np.nan, np.nan, pd.Series(dtype=float)
    session_values = selected.groupby('session')['effect'].mean().dropna()
    if mode == 'grand':
        mean = float(session_values.mean())
        sem = float(session_values.sem()) if len(session_values) > 1 else 0.0
        return mean, sem, session_values
    if mode != 'pooled':
        raise ValueError("mode must be 'pooled' or 'grand'")
    if not precomputed:
        condition_n = int(selected['condition_n'].sum())
        neutral_n = int(selected['neutral_n'].sum())
        condition_p = float(selected['condition_success'].sum() / condition_n)
        neutral_p = float(selected['neutral_success'].sum() / neutral_n)
        mean = (
            condition_p - neutral_p
            if contrast == 'Benefit' else condition_p - neutral_p
        )
        sem = float(np.sqrt(
            condition_p * (1 - condition_p) / condition_n
            + neutral_p * (1 - neutral_p) / neutral_n
        ))
        return mean, sem, session_values
    weights = (selected['condition_n'] + selected['neutral_n']).to_numpy(dtype=float)
    normalized_weights = weights / weights.sum()
    effect_values = selected['effect'].to_numpy(dtype=float)
    mean = float(np.sum(normalized_weights * effect_values))
    weight_square_sum = float(np.sum(normalized_weights ** 2))
    if len(effect_values) > 1 and weight_square_sum < 1:
        weighted_variance = float(
            np.sum(normalized_weights * (effect_values - mean) ** 2)
            / (1 - weight_square_sum)
        )
        sem = float(np.sqrt(weighted_variance * weight_square_sum))
    else:
        sem = 0.0
    return mean, sem, session_values

def context_summary_prediction_pvalue(
    components, contrast, mode, precomputed
):
    easy_mean, easy_sem, easy_sessions = context_summary_category_stats(
        components, contrast, 'Easy', mode, precomputed
    )
    hard_mean, hard_sem, hard_sessions = context_summary_category_stats(
        components, contrast, 'Hard', mode, precomputed
    )
    if mode == 'pooled':
        difference_sem = float(np.sqrt(easy_sem ** 2 + hard_sem ** 2))
        if not np.isfinite(difference_sem) or difference_sem <= 0:
            return np.nan
        z_value = (hard_mean - easy_mean) / difference_sem
        return float(norm.sf(z_value) if contrast == 'Benefit' else norm.cdf(z_value))
    paired = pd.concat(
        [hard_sessions.rename('hard'), easy_sessions.rename('easy')],
        axis=1, join='inner',
    ).dropna()
    if len(paired) < 2:
        return np.nan
    if np.allclose(paired['hard'] - paired['easy'], 0):
        return 1.0
    try:
        alternative = 'greater' if contrast == 'Benefit' else 'less'
        return float(wilcoxon(
            paired['hard'], paired['easy'], alternative=alternative
        ).pvalue)
    except ValueError:
        return np.nan

def format_context_prediction_pvalue(p_value):
    if not np.isfinite(p_value):
        return 'n/a'
    return f'{p_value:.2e}' if p_value < 0.001 else f'{p_value:.3f}'

def plot_context_easy_hard_summary_panel(
    ax, bin_first_components, effect_first_components, mode, title
):
    categories = [
        ('Benefit', 'Easy'), ('Benefit', 'Hard'),
        ('Cost', 'Easy'), ('Cost', 'Hard'),
    ]
    method_specs = [
        ('Bin first', bin_first_components, False, -0.11, 'o'),
        ('Effect first', effect_first_components, True, 0.11, 'D'),
    ]
    pvalue_lines = []
    for method, method_data, precomputed, offset, marker in method_specs:
        means, sems = [], []
        for contrast, difficulty in categories:
            mean, sem, _ = context_summary_category_stats(
                method_data, contrast, difficulty, mode, precomputed
            )
            means.append(mean)
            sems.append(sem)
        x_values = np.arange(len(categories), dtype=float) + offset
        ax.errorbar(
            x_values, means, yerr=sems, marker=marker, linestyle='-',
            markersize=4.5, capsize=2.5, linewidth=1.3,
            color=CONTEXT_SUMMARY_METHOD_COLORS[method], label=method,
        )
        benefit_p = context_summary_prediction_pvalue(
            method_data, 'Benefit', mode, precomputed
        )
        cost_p = context_summary_prediction_pvalue(
            method_data, 'Cost', mode, precomputed
        )
        pvalue_lines.append(
            f'{method}: B H>E p={format_context_prediction_pvalue(benefit_p)}; '
            f'C H<E p={format_context_prediction_pvalue(cost_p)}'
        )
    ax.axhline(0, color='0.45', linestyle=':', linewidth=1)
    ax.axvline(1.5, color='0.75', linestyle='-', linewidth=0.8)
    ax.set_xticks(np.arange(4))
    ax.set_xticklabels(['Benefit\neasy', 'Benefit\nhard', 'Cost\neasy', 'Cost\nhard'])
    ax.set_ylabel('Probability difference (mean ± SEM)')
    ax.set_title(title, fontsize=9.5)
    ax.margins(x=0.08, y=0.28)
    ax.text(
        0.01, 0.99, '\n'.join(pvalue_lines), transform=ax.transAxes,
        ha='left', va='top', fontsize=5.5,
        bbox={'facecolor': 'white', 'edgecolor': '0.8', 'alpha': 0.8, 'pad': 1.5},
    )
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(frameon=False, fontsize=6, loc='best')

def build_context_easy_hard_summary_page(
    bin_first_components, effect_first_components
):
    bin_first_labeled = add_context_effect_difficulty(bin_first_components)
    effect_first_labeled = add_context_effect_difficulty(effect_first_components)
    fig = plt.figure(figsize=(48, 26))
    gs = gridspec.GridSpec(5, 6, figure=fig, hspace=0.68, wspace=0.38)
    for row, (row_label, subject) in enumerate(CONTEXT_EFFECT_ROWS):
        first_row = (
            bin_first_labeled if subject is None
            else bin_first_labeled[bin_first_labeled['subject'].eq(subject)].copy()
        )
        second_row = (
            effect_first_labeled if subject is None
            else effect_first_labeled[effect_first_labeled['subject'].eq(subject)].copy()
        )
        panel_specs = [
            (first_row, second_row, 'pooled', f'{row_label} — pooled — all'),
            (context_effect_subset(first_row, 'Short block'), context_effect_subset(second_row, 'Short block'), 'pooled', f'{row_label} — pooled — short block'),
            (context_effect_subset(first_row, 'Long block'), context_effect_subset(second_row, 'Long block'), 'pooled', f'{row_label} — pooled — long block'),
            (first_row, second_row, 'grand', f'{row_label} — grand average — all'),
            (context_effect_subset(first_row, 'Short block'), context_effect_subset(second_row, 'Short block'), 'grand', f'{row_label} — grand average — short block'),
            (context_effect_subset(first_row, 'Long block'), context_effect_subset(second_row, 'Long block'), 'grand', f'{row_label} — grand average — long block'),
        ]
        for column, (first_panel, second_panel, mode, title) in enumerate(panel_specs):
            plot_context_easy_hard_summary_panel(
                fig.add_subplot(gs[row, column]), first_panel, second_panel, mode, title
            )
    fig.suptitle(
        'PAGE 3 SUMMARY OF BOTH METHODS — Bin first: trials binned before contrasts; Effect first: exact-ISI contrasts calculated before binning\n'
        'DEFINITIONS: Benefit = P(correct | common) − P(correct | neutral); Cost = P(correct | rare) − P(correct | neutral)\n'
        'PREDICTIONS: Benefit_hard > Benefit_easy; Cost_hard < Cost_easy\n'
        'STATISTICS: pooled = one-sided z-test; grand average = paired one-sided Wilcoxon test across sessions',
        fontsize=15, y=0.997,
    )
    fig.subplots_adjust(left=0.055, right=0.99, top=0.94, bottom=0.055)
    return fig

def prepare_common_rare_sequence_data(path=DATA_PATH, n_bins_per_type=4):
    """Prepare outer-bin difficulty labels without breaking raw trial adjacency."""
    available_columns = set(pd.read_csv(path, nrows=0).columns)
    sequence_columns = [
        'subject', 'block_type', 'trial_type', 'trial_isi', 'outcome',
        'session_idx', 'session_uid', 'subject_session_idx',
    ]
    data = pd.read_csv(
        path, usecols=[column for column in sequence_columns if column in available_columns]
    ).copy()
    required = {'subject', 'block_type', 'trial_type', 'trial_isi', 'outcome'}
    missing = required.difference(data.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    data['_source_order'] = np.arange(len(data))
    for column in ['block_type', 'trial_type', 'trial_isi']:
        data[column] = pd.to_numeric(data[column], errors='coerce')
    if 'session_uid' in data.columns:
        data['bias_session'] = data['session_uid'].astype(str)
    elif {'subject', 'subject_session_idx'}.issubset(data.columns):
        data['bias_session'] = (
            data['subject'].astype(str) + '_S' + data['subject_session_idx'].astype(str)
        )
    else:
        data['bias_session'] = data['subject'].astype(str) + '_S' + data['session_idx'].astype(str)
    data['correct'] = np.nan
    data.loc[data['outcome'].eq('Reward'), 'correct'] = 1.0
    data.loc[data['outcome'].eq('Punish'), 'correct'] = 0.0
    data['is_common'] = (
        (data['block_type'].eq(1) & data['trial_type'].eq(1))
        | (data['block_type'].eq(2) & data['trial_type'].eq(2))
    )
    data['is_rare'] = (
        (data['block_type'].eq(1) & data['trial_type'].eq(2))
        | (data['block_type'].eq(2) & data['trial_type'].eq(1))
    )
    valid_for_bins = data[data['outcome'].isin(NON_NAIVE_OUTCOMES)].copy()
    binned_valid = bias_add_psychometric_bins(
        valid_for_bins, n_bins_per_type=n_bins_per_type
    )
    data['psychometric_bin'] = np.nan
    data['binned_difficulty'] = pd.NA
    data.loc[binned_valid.index, 'psychometric_bin'] = binned_valid['psychometric_bin']
    data.loc[binned_valid.index, 'binned_difficulty'] = binned_valid['binned_difficulty']
    return data.sort_values(['bias_session', '_source_order']).copy()

def collect_common_rare_common_sequences(data):
    """Collect adjacent common–outer-bin rare–common triplets in one block."""
    rows = []
    for session, session_data in data.groupby('bias_session', sort=False):
        sdf = session_data.sort_values('_source_order').reset_index(drop=True)
        for anchor_index in range(1, len(sdf) - 1):
            pre = sdf.loc[anchor_index - 1]
            rare = sdf.loc[anchor_index]
            post = sdf.loc[anchor_index + 1]
            if not bool(rare['is_rare']):
                continue
            if not bool(pre['is_common']) or not bool(post['is_common']):
                continue
            if not (pre['block_type'] == rare['block_type'] == post['block_type']):
                continue
            if not (
                pre['_source_order'] + 1 == rare['_source_order']
                and rare['_source_order'] + 1 == post['_source_order']
            ):
                continue
            if any(pd.isna(value) for value in [pre['correct'], rare['correct'], post['correct']]):
                continue
            difficulty = rare['binned_difficulty']
            if pd.isna(difficulty) or difficulty not in ['Easy', 'Difficult']:
                continue
            rows.append({
                'subject': rare['subject'],
                'session': session,
                'sequence_id': f"{session}_T{int(rare['_source_order'])}",
                'block_context': 'Short block' if rare['block_type'] == 1 else 'Long block',
                'difficulty': 'Easy' if difficulty == 'Easy' else 'Hard',
                'anchor_outcome': rare['outcome'],
                'pre': float(pre['correct']),
                'rare': float(rare['correct']),
                'post': float(post['correct']),
                'pre_minus_rare': float(pre['correct'] - rare['correct']),
                'post_minus_rare': float(post['correct'] - rare['correct']),
            })
    return pd.DataFrame(rows)

def sequence_context_subset(events, block_context):
    if block_context is None:
        return events
    return events[events['block_context'].eq(block_context)].copy()

def collect_common_common_common_sequences(data):
    """Collect adjacent common–outer-bin common–common triplets in one block."""
    rows = []
    for session, session_data in data.groupby('bias_session', sort=False):
        sdf = session_data.sort_values('_source_order').reset_index(drop=True)
        for anchor_index in range(1, len(sdf) - 1):
            pre = sdf.loc[anchor_index - 1]
            middle = sdf.loc[anchor_index]
            post = sdf.loc[anchor_index + 1]
            if not all(bool(row['is_common']) for row in [pre, middle, post]):
                continue
            if not (pre['block_type'] == middle['block_type'] == post['block_type']):
                continue
            if not (
                pre['_source_order'] + 1 == middle['_source_order']
                and middle['_source_order'] + 1 == post['_source_order']
            ):
                continue
            if any(pd.isna(value) for value in [pre['correct'], middle['correct'], post['correct']]):
                continue
            difficulty = middle['binned_difficulty']
            if pd.isna(difficulty) or difficulty not in ['Easy', 'Difficult']:
                continue
            rows.append({
                'subject': middle['subject'],
                'session': session,
                'sequence_id': f"{session}_T{int(middle['_source_order'])}",
                'block_context': 'Short block' if middle['block_type'] == 1 else 'Long block',
                'difficulty': 'Easy' if difficulty == 'Easy' else 'Hard',
                'anchor_outcome': middle['outcome'],
                'pre': float(pre['correct']),
                'rare': float(middle['correct']),
                'post': float(post['correct']),
                'pre_minus_rare': float(pre['correct'] - middle['correct']),
                'post_minus_rare': float(post['correct'] - middle['correct']),
            })
    return pd.DataFrame(rows)

def sequence_mean_sem(events, value_column, difficulty, mode):
    selected = events.loc[events['difficulty'].eq(difficulty), ['session', value_column]].dropna()
    if selected.empty:
        return np.nan, np.nan, pd.Series(dtype=float), 0
    if mode == 'pooled':
        values = selected[value_column].astype(float)
        n = len(values)
    elif mode == 'grand':
        values = selected.groupby('session')[value_column].mean().astype(float)
        n = len(values)
    else:
        raise ValueError("mode must be 'pooled' or 'grand'")
    sem = float(values.sem()) if len(values) > 1 else 0.0
    return float(values.mean()), sem, values, n

def sequence_easy_hard_pvalue(events, value_column, mode, binary):
    easy = events[events['difficulty'].eq('Easy')].copy()
    hard = events[events['difficulty'].eq('Hard')].copy()
    if mode == 'pooled':
        easy_values = easy[value_column].dropna().astype(float)
        hard_values = hard[value_column].dropna().astype(float)
        if easy_values.empty or hard_values.empty:
            return np.nan
        if binary:
            table = [
                [int(easy_values.sum()), int(len(easy_values) - easy_values.sum())],
                [int(hard_values.sum()), int(len(hard_values) - hard_values.sum())],
            ]
            return float(fisher_exact(table, alternative='two-sided').pvalue)
        return float(mannwhitneyu(easy_values, hard_values, alternative='two-sided').pvalue)
    paired = pd.concat([
        easy.groupby('session')[value_column].mean().rename('easy'),
        hard.groupby('session')[value_column].mean().rename('hard'),
    ], axis=1, join='inner').dropna()
    if len(paired) < 2:
        return np.nan
    if np.allclose(paired['easy'] - paired['hard'], 0):
        return 1.0
    try:
        return float(wilcoxon(paired['easy'], paired['hard'], alternative='two-sided').pvalue)
    except ValueError:
        return np.nan

def format_sequence_pvalue(p_value):
    if not np.isfinite(p_value):
        return 'n/a'
    return f'{p_value:.2e}' if p_value < 0.001 else f'{p_value:.3f}'

def plot_common_rare_sequence_main(ax, events, mode, title, middle_label='rare'):
    difficulty_specs = [
        ('Easy', SEQUENCE_EASY_COLOR, 0),
        ('Hard', SEQUENCE_HARD_COLOR, 1),
    ]
    for difficulty, color, sequence_x in difficulty_specs:
        phase_x = np.array([sequence_x - 0.18, sequence_x, sequence_x + 0.18])
        phase_means, phase_sems = [], []
        for phase in ['pre', 'rare', 'post']:
            mean, sem, _, _ = sequence_mean_sem(events, phase, difficulty, mode)
            phase_means.append(mean); phase_sems.append(sem)
        ax.plot(phase_x, phase_means, color=color, linewidth=1.2, alpha=0.8)
        for x_value, phase, mean, sem in zip(phase_x, ['pre', 'rare', 'post'], phase_means, phase_sems):
            ax.errorbar(x_value, mean, yerr=sem, fmt=SEQUENCE_PHASE_MARKERS[phase],
                        color=color, markersize=4.5, capsize=2.5, elinewidth=1.2)
    comparison_specs = [('pre', 2), ('rare', 3), ('post', 4)]
    for value_column, x_center in comparison_specs:
        for difficulty, color, offset in [
            ('Easy', SEQUENCE_EASY_COLOR, -0.10), ('Hard', SEQUENCE_HARD_COLOR, 0.10)
        ]:
            mean, sem, _, _ = sequence_mean_sem(events, value_column, difficulty, mode)
            ax.errorbar(x_center + offset, mean, yerr=sem, fmt=SEQUENCE_PHASE_MARKERS[value_column],
                        color=color, markersize=4.5, capsize=2.5, elinewidth=1.2)
        p_value = sequence_easy_hard_pvalue(events, value_column, mode, binary=True)
        ax.text(x_center, 0.98, f'p={format_sequence_pvalue(p_value)}',
                transform=ax.get_xaxis_transform(), ha='center', va='top', fontsize=5.5)
    ax.set_xticks(range(5))
    middle_title = middle_label.capitalize()
    ax.set_xticklabels([
        f'Common–easy {middle_label}–common', f'Common–hard {middle_label}–common',
        'Pre: easy vs hard', f'{middle_title}: easy vs hard', 'Post: easy vs hard',
    ], rotation=18, ha='right')
    ax.set_ylabel('Performance (mean ± SEM)')
    ax.set_title(title, fontsize=9)
    ax.margins(x=0.06, y=0.18)
    ax.grid(False); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    handles = [
        Line2D([0], [0], marker='o', color=SEQUENCE_EASY_COLOR, label='Easy-anchor sequence'),
        Line2D([0], [0], marker='o', color=SEQUENCE_HARD_COLOR, label='Hard-anchor sequence'),
    ]
    ax.legend(handles=handles, frameon=False, fontsize=5.5, loc='best')

def plot_common_rare_sequence_differences(ax, events, mode, middle_label='rare'):
    specs = [
        ('pre_minus_rare', f'Pre − {middle_label}'),
        ('post_minus_rare', f'Post − {middle_label}'),
    ]
    for x_center, (value_column, label) in enumerate(specs):
        for difficulty, color, offset in [
            ('Easy', SEQUENCE_EASY_COLOR, -0.10), ('Hard', SEQUENCE_HARD_COLOR, 0.10)
        ]:
            mean, sem, _, _ = sequence_mean_sem(events, value_column, difficulty, mode)
            ax.errorbar(x_center + offset, mean, yerr=sem, fmt='o', color=color,
                        markersize=4.5, capsize=2.5, elinewidth=1.2)
        p_value = sequence_easy_hard_pvalue(events, value_column, mode, binary=False)
        ax.text(x_center, 0.98, f'p={format_sequence_pvalue(p_value)}',
                transform=ax.get_xaxis_transform(), ha='center', va='top', fontsize=5.5)
    ax.axhline(0, color='0.45', linestyle=':', linewidth=0.9)
    ax.set_xticks([0, 1]); ax.set_xticklabels([label for _, label in specs], rotation=18, ha='right')
    ax.set_ylabel('Performance difference')
    ax.margins(x=0.16, y=0.28)
    ax.grid(False); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

def build_common_rare_sequence_page(
    events, anchor_outcome, page_title, middle_label='rare'
):
    page_events = (
        events if anchor_outcome is None
        else events[events['anchor_outcome'].eq(anchor_outcome)].copy()
    )
    fig = plt.figure(figsize=(70, 27))
    outer = gridspec.GridSpec(5, 6, figure=fig, hspace=0.72, wspace=0.34)
    for row, (row_label, subject) in enumerate(CONTEXT_EFFECT_ROWS):
        row_events = page_events if subject is None else page_events[page_events['subject'].eq(subject)].copy()
        panel_specs = [
            (row_events, 'pooled', f'{row_label} — pooled — all'),
            (sequence_context_subset(row_events, 'Short block'), 'pooled', f'{row_label} — pooled — short block'),
            (sequence_context_subset(row_events, 'Long block'), 'pooled', f'{row_label} — pooled — long block'),
            (row_events, 'grand', f'{row_label} — grand average — all'),
            (sequence_context_subset(row_events, 'Short block'), 'grand', f'{row_label} — grand average — short block'),
            (sequence_context_subset(row_events, 'Long block'), 'grand', f'{row_label} — grand average — long block'),
        ]
        for column, (panel_events, mode, title) in enumerate(panel_specs):
            inner = outer[row, column].subgridspec(1, 2, width_ratios=[3.8, 1.7], wspace=0.32)
            plot_common_rare_sequence_main(
                fig.add_subplot(inner[0, 0]), panel_events, mode, title, middle_label
            )
            plot_common_rare_sequence_differences(
                fig.add_subplot(inner[0, 1]), panel_events, mode, middle_label
            )
    fig.suptitle(
        f'{page_title} — adjacent common–easy/hard {middle_label}–common sequences\n'
        'Points are mean ± SEM; no fixed y limits. Tests: pooled binary performance = two-sided Fisher exact; '
        'pooled differences = two-sided Mann–Whitney U; grand average = paired two-sided Wilcoxon across sessions.',
        fontsize=15, y=0.997,
    )
    fig.subplots_adjust(left=0.035, right=0.995, top=0.94, bottom=0.065)
    return fig

def make_context_effect_figure(
    components, precomputed_components, rare_sequence_events,
    common_sequence_events, output_path=CONTEXT_EFFECT_OUTPUT_PATH
):
    output_path = Path(output_path)
    with PdfPages(output_path) as pdf:
        page_builders = [
            lambda: build_context_effect_page(components),
            lambda: build_precomputed_then_binned_context_effect_page(precomputed_components),
            lambda: build_context_easy_hard_summary_page(components, precomputed_components),
            lambda: build_common_rare_sequence_page(rare_sequence_events, None, 'PAGE 4: All qualifying rare anchors'),
            lambda: build_common_rare_sequence_page(rare_sequence_events, 'Reward', 'PAGE 5: Rewarded rare anchors only'),
            lambda: build_common_rare_sequence_page(rare_sequence_events, 'Punish', 'PAGE 6: Punished rare anchors only'),
            lambda: build_common_rare_sequence_page(common_sequence_events, None, 'PAGE 7: All qualifying common anchors', middle_label='common'),
            lambda: build_common_rare_sequence_page(common_sequence_events, 'Reward', 'PAGE 8: Rewarded common anchors only', middle_label='common'),
            lambda: build_common_rare_sequence_page(common_sequence_events, 'Punish', 'PAGE 9: Punished common anchors only', middle_label='common'),
        ]
        for build_page in page_builders:
            page = build_page()
            pdf.savefig(page, bbox_inches='tight')
            plt.close(page)
            del page
            gc.collect()
    return output_path

if 'bias_data' in globals():
    del bias_data
    gc.collect()
context_source_data = bias_prepare_data()
context_effect_components = collect_context_effect_components(
    context_source_data, n_bins_per_type=4
)
exact_isi_context_effect_components = collect_exact_isi_context_effect_components(
    context_source_data
)
precomputed_then_binned_context_effect_components = bin_precomputed_context_effects(
    exact_isi_context_effect_components, n_bins_per_type=4
)
del context_source_data
gc.collect()
common_rare_sequence_data = prepare_common_rare_sequence_data(
    DATA_PATH, n_bins_per_type=4
)
common_rare_sequence_events = collect_common_rare_common_sequences(
    common_rare_sequence_data
)
common_common_sequence_events = collect_common_common_common_sequences(
    common_rare_sequence_data
)
context_effect_pdf_path = make_context_effect_figure(
    context_effect_components, precomputed_then_binned_context_effect_components,
    common_rare_sequence_events, common_common_sequence_events
)
print(f'Saved: {context_effect_pdf_path.resolve()}')
print(context_effect_components.groupby(
    ['subject', 'contrast', 'block_context']
).agg(sessions=('session', 'nunique'), trials=('condition_n', 'sum')))
print('\nPage 2 exact-ISI contrasts assigned to bins:')
print(precomputed_then_binned_context_effect_components.groupby(
    ['subject', 'contrast', 'block_context']
).agg(sessions=('session', 'nunique'), exact_isi_contrasts=('effect', 'size')))
print('\nPages 4–6 common–rare–common sequences:')
print(common_rare_sequence_events.groupby(
    ['subject', 'anchor_outcome', 'block_context', 'difficulty']
).size().rename('sequences'))
print('\nPages 7–9 common–common–common sequences:')
print(common_common_sequence_events.groupby(
    ['subject', 'anchor_outcome', 'block_context', 'difficulty']
).size().rename('sequences'))


Saved: /home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/context_benefit_cost_by_isi_bin.pdf
                                sessions  trials
subject contrast block_context                  
MC06    Benefit  Long block            1      94
                 Short block           1     101
        Cost     Long block            1      25
                 Short block           1      19

Page 2 exact-ISI contrasts assigned to bins:
                                sessions  exact_isi_contrasts
subject contrast block_context                               
MC06    Benefit  Long block            1                    5
                 Short block           1                    4
        Cost     Long block            1                    4
                 Short block           1                    5

Pages 4–6 common–rare–common sequences:
subject  anchor_outcome  block_context  difficulty
MC06     Punish          Long block  

In [8]:
# Meta-learning: session-by-session context benefit and cost
CONTEXT_META_LEARNING_OUTPUT_PATH = Path('context_benefit_cost_by_session.pdf')
CONTEXT_META_LEARNING_SUBJECTS = ['MC06']
CONTEXT_META_LEARNING_ROLLING_WINDOWS = [3, 5]
CONTEXT_VALUE_ROLLING_WINDOW = 3
CONTEXT_VALUE_COLOR = '#6a3d9a'

def summarize_context_effects_by_session(components):
    """Pool bins and trial types within each session using trial counts."""
    rows = []
    session_order = components[['subject', 'session']].drop_duplicates().copy()
    session_order['session_number'] = (
        session_order.groupby('subject', sort=False).cumcount() + 1
    )
    for (subject, session, contrast), group in components.groupby(
        ['subject', 'session', 'contrast'], sort=False
    ):
        condition_n = int(group['condition_n'].sum())
        neutral_n = int(group['neutral_n'].sum())
        if condition_n == 0 or neutral_n == 0:
            continue
        condition_p = float(group['condition_success'].sum() / condition_n)
        neutral_p = float(group['neutral_success'].sum() / neutral_n)
        rows.append({
            'subject': subject, 'session': session, 'contrast': contrast,
            'effect': condition_p - neutral_p,
            'condition_n': condition_n, 'neutral_n': neutral_n,
        })
    session_effects = pd.DataFrame(rows)
    return session_effects.merge(
        session_order, on=['subject', 'session'], how='left', validate='many_to_one'
    ).sort_values(['subject', 'session_number', 'contrast'])

def align_context_effects_across_subjects(session_effects):
    """Average aligned session numbers after truncating to the shortest history."""
    selected = session_effects[
        session_effects['subject'].isin(CONTEXT_META_LEARNING_SUBJECTS)
    ].copy()
    session_counts = selected.groupby('subject')['session_number'].max()
    missing = set(CONTEXT_META_LEARNING_SUBJECTS).difference(session_counts.index)
    if missing:
        raise ValueError(f'Missing session effects for subjects: {sorted(missing)}')
    common_session_count = int(session_counts.min())
    aligned = selected[selected['session_number'].le(common_session_count)].copy()
    aligned = aligned.groupby(
        ['session_number', 'contrast'], as_index=False
    ).agg(effect=('effect', 'mean'), subjects=('subject', 'nunique'))
    return aligned, common_session_count

def plot_context_meta_learning_panel(
    ax, panel_data, contrasts, title, rolling_window
):
    for contrast in contrasts:
        curve = panel_data[panel_data['contrast'].eq(contrast)].sort_values(
            'session_number'
        )
        if curve.empty:
            continue
        rolling_mean = curve['effect'].rolling(
            window=rolling_window, center=True, min_periods=1
        ).mean()
        color = CONTEXT_EFFECT_COLORS[contrast]
        ax.scatter(
            curve['session_number'], curve['effect'], s=18, alpha=0.38,
            color=color, label=f'{contrast}: session value', zorder=2,
        )
        ax.plot(
            curve['session_number'], rolling_mean, color=color, linewidth=2.2,
            label=f'{contrast}: {rolling_window}-session rolling mean',
            zorder=3,
        )
    ax.axhline(0, color='0.45', linestyle=':', linewidth=1)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Session number (oldest → latest)')
    ax.set_ylabel('Probability difference')
    ax.margins(x=0.02, y=0.14)
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

def build_context_meta_learning_figure(session_effects, rolling_window, page_number):
    aligned_effects, common_session_count = align_context_effects_across_subjects(
        session_effects
    )
    rows = [('All subjects', None)] + [
        (subject, subject) for subject in CONTEXT_META_LEARNING_SUBJECTS
    ]
    columns = [
        (['Benefit', 'Cost'], 'Benefit and cost'),
        (['Benefit'], 'Benefit'),
        (['Cost'], 'Cost'),
    ]
    fig, axes = plt.subplots(5, 3, figsize=(22, 24), squeeze=False)
    for row_index, (row_label, subject) in enumerate(rows):
        row_data = (
            aligned_effects if subject is None
            else session_effects[session_effects['subject'].eq(subject)].copy()
        )
        for column_index, (contrasts, column_label) in enumerate(columns):
            title = (
                f'{row_label} — {column_label} — first {common_session_count} aligned sessions'
                if subject is None else f'{row_label} — {column_label}'
            )
            plot_context_meta_learning_panel(
                axes[row_index, column_index], row_data, contrasts, title, rolling_window
            )
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.962),
        ncol=4, frameon=False, fontsize=9,
    )
    fig.suptitle(
        f'PAGE {page_number}: Meta-learning across sessions — {rolling_window}-session rolling mean (chronological order)\n'
        'Benefit = P(correct | common) − P(correct | neutral); '
        'Cost = P(correct | rare) − P(correct | neutral)',
        fontsize=16, y=0.997,
    )
    fig.subplots_adjust(left=0.065, right=0.985, top=0.925, bottom=0.045, hspace=0.42, wspace=0.28)
    return fig, common_session_count

def calculate_context_value_by_session(session_effects):
    """Calculate V_context from session-specific common/rare trial probabilities."""
    effect_wide = session_effects.pivot(
        index=['subject', 'session', 'session_number'],
        columns='contrast', values=['effect', 'condition_n'],
    ).dropna().reset_index()
    effect_wide.columns = [
        '_'.join(str(part) for part in column if str(part))
        if isinstance(column, tuple) else str(column)
        for column in effect_wide.columns
    ]
    total_context_trials = (
        effect_wide['condition_n_Benefit'] + effect_wide['condition_n_Cost']
    )
    effect_wide['p_common'] = effect_wide['condition_n_Benefit'] / total_context_trials
    effect_wide['p_rare'] = effect_wide['condition_n_Cost'] / total_context_trials
    effect_wide['rare_penalty'] = effect_wide['effect_Cost']
    effect_wide['V_context'] = (
        effect_wide['p_common'] * effect_wide['effect_Benefit']
        + effect_wide['p_rare'] * effect_wide['rare_penalty']
    )
    return effect_wide.sort_values(['subject', 'session_number'])

def align_context_values_across_subjects(context_values, common_session_count):
    selected = context_values[
        context_values['subject'].isin(CONTEXT_META_LEARNING_SUBJECTS)
        & context_values['session_number'].le(common_session_count)
    ].copy()
    return selected.groupby('session_number', as_index=False).agg(
        V_context=('V_context', 'mean'),
        p_common=('p_common', 'mean'), p_rare=('p_rare', 'mean'),
        subjects=('subject', 'nunique'),
    )

def plot_context_value_panel(ax, panel_data, title):
    curve = panel_data.sort_values('session_number')
    rolling_mean = curve['V_context'].rolling(
        window=CONTEXT_VALUE_ROLLING_WINDOW, center=True, min_periods=1
    ).mean()
    ax.scatter(
        curve['session_number'], curve['V_context'], s=22, alpha=0.42,
        color=CONTEXT_VALUE_COLOR, label='Session V_context', zorder=2,
    )
    ax.plot(
        curve['session_number'], rolling_mean, color=CONTEXT_VALUE_COLOR,
        linewidth=2.4, label=f'{CONTEXT_VALUE_ROLLING_WINDOW}-session rolling mean',
        zorder=3,
    )
    ax.axhline(0, color='0.45', linestyle=':', linewidth=1)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Session number (oldest → latest)')
    ax.set_ylabel('V_context')
    ax.margins(x=0.02, y=0.14)
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc='best')

def build_context_value_figure(context_values, common_session_count):
    aligned_values = align_context_values_across_subjects(
        context_values, common_session_count
    )
    rows = [('All subjects', None)] + [
        (subject, subject) for subject in CONTEXT_META_LEARNING_SUBJECTS
    ]
    fig, axes = plt.subplots(5, 1, figsize=(16, 24), squeeze=False)
    for row_index, (row_label, subject) in enumerate(rows):
        row_data = (
            aligned_values if subject is None
            else context_values[context_values['subject'].eq(subject)].copy()
        )
        title = (
            f'{row_label} — first {common_session_count} aligned sessions'
            if subject is None else row_label
        )
        plot_context_value_panel(axes[row_index, 0], row_data, title)
    fig.suptitle(
        'PAGE 3: Session-level context value (chronological order)\n'
        'V_context = p_common × Benefit + p_rare × Rare penalty; '
        'Rare penalty = Cost = P(correct | rare) − P(correct | neutral)',
        fontsize=15, y=0.997,
    )
    fig.subplots_adjust(left=0.085, right=0.985, top=0.945, bottom=0.045, hspace=0.42)
    return fig

context_session_effects = summarize_context_effects_by_session(
    context_effect_components
)
_, context_common_session_count = align_context_effects_across_subjects(
    context_session_effects
)
context_session_values = calculate_context_value_by_session(
    context_session_effects
)
with PdfPages(CONTEXT_META_LEARNING_OUTPUT_PATH) as pdf:
    for page_number, rolling_window in enumerate(
        CONTEXT_META_LEARNING_ROLLING_WINDOWS, start=1
    ):
        page, _ = build_context_meta_learning_figure(
            context_session_effects, rolling_window, page_number
        )
        pdf.savefig(page, bbox_inches='tight')
        plt.close(page)
    context_value_page = build_context_value_figure(
        context_session_values, context_common_session_count
    )
    pdf.savefig(context_value_page, bbox_inches='tight')
    plt.close(context_value_page)
print(f'Saved: {CONTEXT_META_LEARNING_OUTPUT_PATH.resolve()}')
print(f'All-subject row uses the first {context_common_session_count} sessions per subject.')


Saved: /home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/context_benefit_cost_by_session.pdf
All-subject row uses the first 1 sessions per subject.


In [9]:
# Rewarded fraction across the same four within-session psychometric bins
REWARDED_BIN_OUTPUT_PATH = Path('rewarded_fraction_by_psychometric_bin.pdf')
REWARDED_BIN_COLUMNS = [
    ('All formal blocks', 'formal'),
    ('Short block', 1),
    ('Long block', 2),
    ('Neutral, short, and long blocks', 'overlay'),
]

def prepare_rewarded_bin_data(path=DATA_PATH, n_bins_per_type=4):
    data = pd.read_csv(path).copy()
    required = {'subject', 'block_type', 'trial_type', 'trial_isi', 'outcome'}
    missing = required.difference(data.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    for column in ['block_type', 'trial_type', 'trial_isi']:
        data[column] = pd.to_numeric(data[column], errors='coerce')
    data = data[
        data['block_type'].isin([0, 1, 2])
        & data['trial_type'].isin([1, 2])
        & data['outcome'].isin(VALID_OUTCOMES)
    ].dropna(subset=['trial_isi']).copy()
    data['rewarded'] = data['outcome'].isin(CORRECT_OUTCOMES).astype(float)
    if 'session_uid' in data.columns:
        data['reward_bin_session'] = data['session_uid'].astype(str)
    elif 'subject_session_idx' in data.columns:
        data['reward_bin_session'] = (
            data['subject'].astype(str) + '_S' + data['subject_session_idx'].astype(str)
        )
    else:
        data['reward_bin_session'] = (
            data['subject'].astype(str) + '_S' + data['session_idx'].astype(str)
        )
    data['psychometric_bin'] = np.nan
    for _, session_data in data.groupby('reward_bin_session', sort=False):
        for trial_type in [1, 2]:
            trial_index = session_data.index[session_data['trial_type'].eq(trial_type)]
            if len(trial_index) == 0:
                continue
            # Match the seeded, trial-type-specific four-bin psychometric logic.
            np.random.seed(42)
            jittered_isi = (
                data.loc[trial_index, 'trial_isi']
                + np.random.normal(0, 1e-6, len(trial_index))
            )
            try:
                bin_codes = pd.qcut(
                    jittered_isi, q=n_bins_per_type, labels=False, duplicates='drop'
                )
            except ValueError:
                continue
            data.loc[trial_index, 'psychometric_bin'] = bin_codes.to_numpy(dtype=float)
    return data.dropna(subset=['psychometric_bin']).copy()

def rewarded_bin_stats(data, trial_type, n_bins_per_type=4):
    rows = []
    for bin_index in range(n_bins_per_type):
        values = data.loc[
            data['trial_type'].eq(trial_type)
            & data['psychometric_bin'].eq(bin_index), 'rewarded'
        ]
        n = len(values)
        fraction = float(values.mean()) if n else np.nan
        sem = np.sqrt(fraction * (1 - fraction) / n) if n else np.nan
        rows.append({'bin': bin_index + 1, 'fraction': fraction, 'sem': sem, 'n': n})
    return pd.DataFrame(rows)

def plot_rewarded_bin_panel(ax, panel, title, n_bins_per_type=4):
    positions = {1: np.arange(n_bins_per_type), 2: np.arange(n_bins_per_type) + 5}
    colors = {1: SHORT_COLOR, 2: LONG_COLOR}
    labels = {1: 'Short trials', 2: 'Long trials'}
    handles = []
    for trial_type in [1, 2]:
        stats = rewarded_bin_stats(panel, trial_type, n_bins_per_type)
        valid = stats['fraction'].notna()
        ax.errorbar(
            positions[trial_type][valid], stats.loc[valid, 'fraction'],
            yerr=stats.loc[valid, 'sem'], color=colors[trial_type],
            marker='o', linewidth=1.8, markersize=5.5, capsize=3.5
        )
        handles.append(Line2D(
            [0], [0], color=colors[trial_type], marker='o',
            label=f"{labels[trial_type]} (n={int(stats['n'].sum()):,})"
        ))
    ax.set_title(title, fontsize=11)
    ax.set_xticks(
        np.r_[positions[1], positions[2]],
        [f'S{i}' for i in range(1, n_bins_per_type + 1)]
        + [f'L{i}' for i in range(1, n_bins_per_type + 1)]
    )
    ax.set_xlabel('Psychometric ISI bin')
    ax.set_ylabel('Fraction rewarded')
    ax.margins(y=0.15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(handles=handles, frameon=False, fontsize=7, loc='best')

def plot_rewarded_block_overlay_panel(ax, panel, title, n_bins_per_type=4):
    positions = {1: np.arange(n_bins_per_type), 2: np.arange(n_bins_per_type) + 5}
    block_colors = {0: '0.45', 1: '#BCAE3E', 2: 'blue'}
    block_labels = {0: 'Neutral block', 1: 'Short block', 2: 'Long block'}
    handles = []
    for block_type in [0, 1, 2]:
        block_data = panel[panel['block_type'].eq(block_type)]
        for trial_type in [1, 2]:
            stats = rewarded_bin_stats(block_data, trial_type, n_bins_per_type)
            valid = stats['fraction'].notna()
            ax.errorbar(
                positions[trial_type][valid], stats.loc[valid, 'fraction'],
                yerr=stats.loc[valid, 'sem'], color=block_colors[block_type],
                marker='o', linewidth=1.8, markersize=5, capsize=3
            )
        handles.append(Line2D(
            [0], [0], color=block_colors[block_type], marker='o',
            label=f"{block_labels[block_type]} (n={len(block_data):,})"
        ))
    ax.set_title(title, fontsize=11)
    ax.set_xticks(
        np.r_[positions[1], positions[2]],
        [f'S{i}' for i in range(1, n_bins_per_type + 1)]
        + [f'L{i}' for i in range(1, n_bins_per_type + 1)]
    )
    ax.set_xlabel('Psychometric ISI bin')
    ax.set_ylabel('Fraction rewarded')
    ax.margins(y=0.15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(handles=handles, frameon=False, fontsize=7, loc='best')

def make_rewarded_bin_figure(data, output_path=REWARDED_BIN_OUTPUT_PATH, n_bins_per_type=4):
    fig, axes = plt.subplots(5, 4, figsize=(24, 24), sharex=True, sharey=False)
    for row_index, (row_label, subject) in enumerate(SUBJECT_ROWS):
        row_data = data if subject is None else data[data['subject'].eq(subject)]
        for column_index, (column_label, column_key) in enumerate(REWARDED_BIN_COLUMNS):
            if column_key == 'formal':
                panel = row_data[row_data['block_type'].isin([1, 2])]
            elif column_key == 'overlay':
                panel = row_data
            else:
                panel = row_data[row_data['block_type'].eq(column_key)]
            plotter = (
                plot_rewarded_block_overlay_panel
                if column_key == 'overlay' else plot_rewarded_bin_panel
            )
            plotter(axes[row_index, column_index], panel,
                    f'{row_label} — {column_label}', n_bins_per_type=n_bins_per_type)
    fig.suptitle(
        'Rewarded fraction by four-bin psychometric ISI category', fontsize=16, y=0.995
    )
    fig.tight_layout(rect=[0.025, 0.02, 0.995, 0.98], h_pad=2.0, w_pad=1.2)
    output_path = Path(output_path)
    fig.savefig(output_path, bbox_inches='tight')
    plt.show()
    return output_path

rewarded_bin_data = prepare_rewarded_bin_data()
rewarded_bin_pdf_path = make_rewarded_bin_figure(rewarded_bin_data)
print(f'Saved: {rewarded_bin_pdf_path.resolve()}')


Saved: /home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/rewarded_fraction_by_psychometric_bin.pdf


In [10]:
# Endpoint versions of block-segment report pages 5 and 8: trials 5–11 vs last 7 trials
from matplotlib.backends.backend_pdf import PdfPages

ENDPOINT_OUTPUT_PATH = Path('block_endpoint_trials_5_to_11_vs_last_7_pages_5_8.pdf')
ENDPOINT_WINDOW_SIZE = 7
ENDPOINT_ORDER = ['Trials 5–11', 'Last 7 trials']
ENDPOINT_X = dict(zip(ENDPOINT_ORDER, range(len(ENDPOINT_ORDER))))
ENDPOINT_PHASES = ['pre', 'rare', 'post']
ENDPOINT_PHASE_LABELS = {'pre': 'Pre rare', 'rare': 'Rare', 'post': 'Post rare'}
ENDPOINT_PHASE_COLORS = {'pre': '#0072B2', 'rare': '#D55E00', 'post': '#009E73'}

def collect_endpoint_window_events(events, window_size=ENDPOINT_WINDOW_SIZE):
    required = {
        'block_length', 'train_start_position', 'train_end_position',
        'block_context', 'subject', 'session',
    }
    missing = required.difference(events.columns)
    if missing:
        raise ValueError(f'Missing endpoint-event columns: {sorted(missing)}')
    eligible = events[events['block_length'].ge(window_size)].copy()
    endpoint_frames = []
    for endpoint, window_start in [
        ('Trials 5–11', pd.Series(4, index=eligible.index, dtype=int)),
        ('Last 7 trials', eligible['block_length'].astype(int) - window_size),
    ]:
        window_stop = window_start + window_size
        contained = (
            window_start.ge(0)
            & window_stop.le(eligible['block_length'])
            & eligible['train_start_position'].ge(window_start)
            & eligible['train_end_position'].lt(window_stop)
        )
        selected = eligible.loc[contained].copy()
        selected['endpoint'] = endpoint
        selected['window_start'] = window_start.loc[selected.index].astype(int)
        selected['window_stop'] = window_stop.loc[selected.index].astype(int)
        endpoint_frames.append(selected)
    result = pd.concat(endpoint_frames, ignore_index=True) if endpoint_frames else pd.DataFrame()
    if not result.empty:
        bad = result[
            result['window_start'].lt(0)
            | result['window_stop'].gt(result['block_length'])
            | result['train_start_position'].lt(result['window_start'])
            | result['train_end_position'].ge(result['window_stop'])
        ]
        if not bad.empty:
            raise AssertionError('An endpoint window or qualifying train crossed a block boundary')
    return result

def endpoint_mean_sem(events, value_column, endpoint, mode):
    selected = events[events['endpoint'].eq(endpoint)]
    if mode == 'pooled':
        values = selected[value_column].dropna().astype(float)
    elif mode == 'grand':
        values = selected.groupby('session')[value_column].mean().dropna().astype(float)
    else:
        raise ValueError("mode must be 'pooled' or 'grand'")
    if values.empty:
        return np.nan, np.nan, 0
    sem = float(values.sem()) if len(values) > 1 else 0.0
    return float(values.mean()), sem, len(values)

def style_endpoint_axis(ax, title):
    ax.set_xticks(range(len(ENDPOINT_ORDER)), ENDPOINT_ORDER)
    ax.set_xlim(-0.25, len(ENDPOINT_ORDER) - 0.75)
    ax.set_xlabel('Position within adjusted block')
    ax.set_ylabel('Performance (fraction correct)')
    ax.set_title(title, fontsize=9)
    ax.margins(y=0.15)
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

def plot_sequence_endpoint_panel(ax, events, mode, title):
    for phase in ENDPOINT_PHASES:
        means, sems = [], []
        for endpoint in ENDPOINT_ORDER:
            mean, sem, _ = endpoint_mean_sem(events, phase, endpoint, mode)
            means.append(mean)
            sems.append(sem)
        ax.errorbar(
            range(len(ENDPOINT_ORDER)), means, yerr=sems, marker='o',
            markersize=4.5, capsize=2.5, linewidth=1.5, elinewidth=1.0,
            color=ENDPOINT_PHASE_COLORS[phase], alpha=0.70,
            label=ENDPOINT_PHASE_LABELS[phase],
        )
    style_endpoint_axis(ax, title)
    ax.legend(frameon=False, fontsize=6, loc='best')

def plot_previous_outcome_endpoint_panel(ax, events, mode, title):
    trace_specs = [
        ('rewarded_common', 'Previous common rewarded', '#009E73', 'o'),
        ('punished_common', 'Previous common punished', '#D55E00', 's'),
    ]
    for trace, label, color, marker in trace_specs:
        trace_events = events[events['trace'].eq(trace)]
        means, sems = [], []
        for endpoint in ENDPOINT_ORDER:
            mean, sem, _ = endpoint_mean_sem(trace_events, 'performance', endpoint, mode)
            means.append(mean)
            sems.append(sem)
        ax.errorbar(
            range(len(ENDPOINT_ORDER)), means, yerr=sems, marker=marker,
            markersize=4.5, capsize=2.5, linewidth=1.5, elinewidth=1.0,
            color=color, alpha=0.70, label=label,
        )
    style_endpoint_axis(ax, title)
    ax.legend(frameon=False, fontsize=6, loc='best')

def endpoint_six_panel_specs(row_events):
    short_events = row_events[row_events['block_context'].eq('Short block')]
    long_events = row_events[row_events['block_context'].eq('Long block')]
    return [
        (row_events, 'pooled', 'pooled events'),
        (row_events, 'grand', 'grand average across sessions'),
        (short_events, 'pooled', 'short block — pooled events'),
        (short_events, 'grand', 'short block — grand average'),
        (long_events, 'pooled', 'long block — pooled events'),
        (long_events, 'grand', 'long block — grand average'),
    ]

def build_endpoint_page(events, plot_function, page_title):
    fig = plt.figure(figsize=(66, 27))
    gs = gridspec.GridSpec(5, 6, figure=fig, hspace=0.60, wspace=0.52)
    for row, (row_label, subject) in enumerate(SUBJECT_ROWS):
        row_events = events if subject is None else events[events['subject'].eq(subject)].copy()
        for column, (panel_events, mode, suffix) in enumerate(endpoint_six_panel_specs(row_events)):
            plot_function(
                fig.add_subplot(gs[row, column]), panel_events, mode,
                f'{row_label} — {suffix}',
            )
    fig.suptitle(page_title, fontsize=16, y=0.995)
    fig.subplots_adjust(left=0.035, right=0.985, top=0.972, bottom=0.055)
    return fig

required_endpoint_globals = ['block_sequence_events', 'block_previous_outcome_events']
missing_endpoint_globals = [name for name in required_endpoint_globals if name not in globals()]
if missing_endpoint_globals:
    raise RuntimeError(
        'Run the block-segment performance cell first; missing: '
        + ', '.join(missing_endpoint_globals)
    )

sequence_endpoint_events = collect_endpoint_window_events(block_sequence_events)
previous_outcome_endpoint_events = collect_endpoint_window_events(
    block_previous_outcome_events
)

sequence_endpoint_page = build_endpoint_page(
    sequence_endpoint_events, plot_sequence_endpoint_panel,
    'Page 5 endpoint analysis: common/common/rare/common performance\n'
    'trials 5–11 versus last 7 trials of each adjusted block',
)
previous_outcome_endpoint_page = build_endpoint_page(
    previous_outcome_endpoint_events, plot_previous_outcome_endpoint_panel,
    'Page 8 endpoint analysis: common-trial performance after rewarded versus punished common trials\n'
    'trials 5–11 versus last 7 trials of each adjusted block',
)

with PdfPages(ENDPOINT_OUTPUT_PATH) as pdf:
    for page in [sequence_endpoint_page, previous_outcome_endpoint_page]:
        pdf.savefig(page, bbox_inches='tight')
        plt.close(page)

print(f'Saved: {ENDPOINT_OUTPUT_PATH.resolve()}')
print('Page 5 endpoint events:')
print(sequence_endpoint_events.groupby(['subject', 'endpoint']).size().unstack(fill_value=0))
print('\nPage 8 endpoint events:')
print(previous_outcome_endpoint_events.groupby(['subject', 'endpoint']).size().unstack(fill_value=0))


Saved: /home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/block_endpoint_trials_5_to_11_vs_last_7_pages_5_8.pdf
Page 5 endpoint events:
endpoint  Last 7 trials  Trials 5–11
subject                             
MC06                  3            5

Page 8 endpoint events:
endpoint  Last 7 trials  Trials 5–11
subject                             
MC06                 41           28
